# Probing XLM-R Representations for Morphosyntactic Generalization to Held-Out Lemmata in English, Italian, and Polish

## 1. Environment and Reproducibility

Pins the environment (Python 3.11, `requirements.txt`, the UD release recorded in `data/ud/MANIFEST.json`) and writes `config.json` and `results/run_manifest.json`.

In [1]:
import sys
import platform

assert sys.version_info[:2] == (3, 11), f"the pinned environment is Python 3.11, got {platform.python_version()}"

import conllu
import numpy as np
import pandas as pd
import torch

from importlib.metadata import version

STACK = ("torch", "transformers", "tokenizers", "scikit-learn", "numpy", "scipy",
         "statsmodels", "conllu", "pandas", "matplotlib")

print(f"python        {platform.python_version()}  ({platform.platform()})")
for pkg in STACK:
    print(f"{pkg:<13} {version(pkg)}")
print(f"cuda          {torch.cuda.is_available()}  (CPU-only setup is expected: inference-only + linear probes)")

python        3.11.9  (Windows-10-10.0.26200-SP0)
torch         2.13.0+cpu
transformers  5.14.1
tokenizers    0.22.2
scikit-learn  1.9.0
numpy         2.4.6
scipy         1.17.1
statsmodels   0.14.6
conllu        6.0.0
pandas        3.0.3
matplotlib    3.11.1
cuda          False  (CPU-only setup is expected: inference-only + linear probes)


In [2]:
import random

SEED = 42   # one global seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_default_dtype(torch.float32)
torch.set_grad_enabled(False)

print(f"SEED = {SEED}; torch deterministic = {torch.are_deterministic_algorithms_enabled()}; "
      f"grad enabled = {torch.is_grad_enabled()}")

SEED = 42; torch deterministic = True; grad enabled = False


In [3]:
import json
from pathlib import Path

PROJECT = Path.cwd()
UD_DIR = PROJECT / "data" / "ud"
CACHE_DIR = PROJECT / "cache"
REPR_DIR = CACHE_DIR / "representations"
SPLIT_DIR = CACHE_DIR / "splits"
PRED_DIR = CACHE_DIR / "predictions"
RESULTS_DIR = PROJECT / "results"
FIG_DIR = RESULTS_DIR / "figures"

for d in (REPR_DIR, SPLIT_DIR, PRED_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# MANIFEST.json ships with the repository; download_ud.py fetches the files it lists
ud_files = [f for tb in json.loads((UD_DIR / "MANIFEST.json").read_text(encoding="utf-8"))["treebanks"].values()
            for f in tb["files"]]
assert all((UD_DIR / f).exists() for f in ud_files), "run download_ud.py first"
print("\n".join(str(d.relative_to(PROJECT)) for d in (UD_DIR, REPR_DIR, SPLIT_DIR, PRED_DIR, RESULTS_DIR, FIG_DIR)))

data\ud
cache\representations
cache\splits
cache\predictions
results
results\figures


In [4]:
import json

CONFIG = {
    "seed": SEED,

    "model_name": "xlm-roberta-base",
    "layer_index": 8,
    "readout": "last_subword",
    "dtype": "fp32",
    "max_length": 512,
    "tokenizer": {"fast": True, "add_special_tokens": True, "return_offsets_mapping": True},

    "treebanks": {"UD_English-EWT": "en_ewt", "UD_Italian-ISDT": "it_isdt", "UD_Polish-PDB": "pl_pdb"},
    "split_usage": "pool_native_splits",
    "tasks": {
        "upos":   {"label": "UPOS", "classes": "all_observed"},
        "number": {"label": "Number", "classes": ["Sing", "Plur"], "upos_filter": "NOUN"},
    },
    "token_eligibility": {
        "exclude_mwt_component_words": True,
        "exclude_mwt_range_tokens": True,
        "exclude_empty_nodes": True,
        "exclude_missing_lemma": True,
    },

    "split": {
        "test_fraction_target": 0.20,
        "min_heldout_lemmas": 50,
        "per_lemma_test_seen": [0.5, 0.5],
        "size_matching": "tight",
    },

    "standardization": "StandardScaler_per_condition_fit_on_train",
    "probe": {"estimator": "LogisticRegression", "penalty": "l2", "C": 1.0, "solver": "lbfgs",
              "max_iter": 2000, "tol": "default", "class_weight": None, "random_state": SEED},

    "control": {"labels": "random_per_lemma_type_from_empirical_marginal", "assignment": "once_globally"},

    "evaluation": {"bootstrap_resamples": 10_000, "ci_percentiles": [2.5, 97.5],
                   "mcnemar": "exact_if_b_plus_c_small_else_chi2_continuity",
                   "holm_correction": "reported_alongside_raw_p"},

    "error_analysis": {"predictors": ["n_subwords", "char_length", "log_freq", "C(label)"],
                       "log_freq": "ln_form_frequency_pooled_treebank_add_one",
                       "zscore_continuous": True, "scope": "per_language_task", "report_vif": True},

    "feasibility": {"min_heldout_lemmas": 50, "min_test_instances": 200,
                    "min_per_class_number": 100, "min_contributing_heldout_lemmas": 30},

    "extensions": {"mlp_hidden_units": 256, "layer_sweep": [4, 8, 11]},
}

CONFIG_PATH = PROJECT / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(f"wrote {CONFIG_PATH.name} ({len(json.dumps(CONFIG))} bytes)")

wrote config.json (1731 bytes)


In [5]:
from datetime import datetime, timezone

ud_manifest = json.loads((UD_DIR / "MANIFEST.json").read_text(encoding="utf-8"))

# pinned revision is only confirmed against the Hub or the local cache, never resolved
from huggingface_hub import HfApi, scan_cache_dir

MODEL_REVISION_PIN = "e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
try:
    model_revision = HfApi().model_info(CONFIG["model_name"], revision=MODEL_REVISION_PIN).sha
    revision_source = "hub"
except Exception:
    model_revision = next((rev.commit_hash
                           for repo in scan_cache_dir().repos if repo.repo_id == CONFIG["model_name"]
                           for rev in repo.revisions if rev.commit_hash == MODEL_REVISION_PIN), None)
    revision_source = "local_cache"
assert model_revision == MODEL_REVISION_PIN, (
    f"pinned revision {MODEL_REVISION_PIN[:12]} of {CONFIG['model_name']} found neither on the Hub "
    f"nor in the local HF cache")

RUN_MANIFEST = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {pkg: version(pkg) for pkg in STACK},
    "requirements_frozen": "requirements.txt",
    "seed": SEED,
    "ud_release": ud_manifest["ud_release"],
    "ud_downloaded": ud_manifest["downloaded"],
    "treebank_commits": {repo: info["commit"] for repo, info in ud_manifest["treebanks"].items()},
    "data_sha256": {name: sha for info in ud_manifest["treebanks"].values()
                    for name, sha in info["files"].items()},
    "model_name": CONFIG["model_name"],
    "model_revision": model_revision,
    "model_revision_source": revision_source,
    "config": CONFIG,
}

MANIFEST_PATH = RESULTS_DIR / "run_manifest.json"
MANIFEST_PATH.write_text(json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8")

print(f"UD release        {RUN_MANIFEST['ud_release']} (downloaded {RUN_MANIFEST['ud_downloaded']})")
for repo, sha in RUN_MANIFEST["treebank_commits"].items():
    print(f"  {repo:<18} {sha[:12]}")
print(f"model             {RUN_MANIFEST['model_name']} @ {model_revision[:12]} ({revision_source})")
print(f"wrote             {MANIFEST_PATH.relative_to(PROJECT)}")

UD release        r2.18 (downloaded 2026-07-22)
  UD_English-EWT     b7711cce01cd
  UD_Italian-ISDT    4852011b996b
  UD_Polish-PDB      96706b419ad3
model             xlm-roberta-base @ e73636d4f797 (hub)
wrote             results\run_manifest.json


## 2. Model and Readout

Loads `xlm-roberta-base` frozen at the revision recorded in the run manifest, checks the layer-indexing convention, and implements the readout.

In [6]:
from transformers import AutoModel, AutoTokenizer

MODEL_REVISION = RUN_MANIFEST["model_revision"]

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], revision=MODEL_REVISION)
assert tokenizer.is_fast, "offset mapping (input construction) requires the fast tokenizer"

model = AutoModel.from_pretrained(CONFIG["model_name"], revision=MODEL_REVISION,
                                  output_hidden_states=True)
model.eval()
model.requires_grad_(False)

assert model.config.num_hidden_layers == 12 and model.config.hidden_size == 768
assert model.dtype == torch.float32
print(f"{CONFIG['model_name']} @ {MODEL_REVISION[:12]}: "
      f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M params, frozen "
      f"(training={model.training}, any grad={any(p.requires_grad for p in model.parameters())})")

xlm-roberta-base @ e73636d4f797: 278M params, frozen (training=False, any grad=False)


In [7]:
# layer-indexing convention
with torch.no_grad():
    probe_out = model(**tokenizer("Kot śpi.", return_tensors="pt"))
hs = probe_out.hidden_states

assert len(hs) == 13, f"expected 13 hidden-state tensors, got {len(hs)}"
assert all(t.shape == hs[0].shape for t in hs) and hs[0].shape[-1] == 768
assert not torch.equal(hs[0], hs[CONFIG["layer_index"]]), "layer 8 must differ from embeddings"

print(f"hidden_states: {len(hs)} tensors of shape {tuple(hs[0].shape)} "
      f"(0 = embeddings, 1..12 = transformer blocks)")
print(f"layer {CONFIG['layer_index']} = hidden_states[{CONFIG['layer_index']}]  ✓")

hidden_states: 13 tensors of shape (1, 5, 768) (0 = embeddings, 1..12 = transformer blocks)
layer 8 = hidden_states[8]  ✓


In [8]:
def encode_sentence(text: str):
    """Tokenize and run one sentence; return the offsets and the layer-8 states."""
    enc = tokenizer(text, add_special_tokens=True, truncation=True,
                    max_length=CONFIG["max_length"], return_offsets_mapping=True,
                    return_tensors="pt")
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    offsets = enc["offset_mapping"][0].tolist()
    layer = out.hidden_states[CONFIG["layer_index"]][0]
    return offsets, layer


def subwords_for_span(offsets, start: int, end: int) -> list[int]:
    """Indices of the subwords whose character spans overlap [start, end)."""
    return [i for i, (s, e) in enumerate(offsets) if s < end and e > start and e > s]


def last_subword_vector(offsets, layer, start: int, end: int):
    """The last overlapping subword's layer-8 state, and the subword count."""
    idx = subwords_for_span(offsets, start, end)
    if not idx:
        return None, 0
    return layer[idx[-1]], len(idx)


# sanity checks on toy sentences
sent = "The unfriendliest cats sleep."
offsets, layer = encode_sentence(sent)
for word in ("The", "unfriendliest", "cats", "sleep"):
    start = sent.index(word); end = start + len(word)
    vec, n_sub = last_subword_vector(offsets, layer, start, end)
    assert vec is not None and vec.shape == (768,) and vec.dtype == torch.float32
    print(f"  {word:<14} n_subwords={n_sub}  vec=(768,) fp32  ✓")

# final word dropped for sentence over 512 subwords
long_sent = " ".join(["antidisestablishmentarianism"] * 100)
offsets_l, layer_l = encode_sentence(long_sent)
vec, n_sub = last_subword_vector(offsets_l, layer_l,
                                 len(long_sent) - len("antidisestablishmentarianism"), len(long_sent))
assert vec is None and n_sub == 0 and layer_l.shape[0] == CONFIG["max_length"]
print(f"  truncation: seq capped at {layer_l.shape[0]}; out-of-window target -> dropped & logged  ✓")

  The            n_subwords=1  vec=(768,) fp32  ✓
  unfriendliest  n_subwords=4  vec=(768,) fp32  ✓
  cats           n_subwords=2  vec=(768,) fp32  ✓
  sleep          n_subwords=1  vec=(768,) fp32  ✓
  truncation: seq capped at 512; out-of-window target -> dropped & logged  ✓


## 3. Data and Task Inventories

Parses pinned CoNLL-U files, pools each treebank's native splits, applies eligibility rules, and builds UPOS and Number inventories with full exclusion counts. Writes `cache/inventory/` and `results/inventory_report.json`.

In [9]:
def iter_pooled_sentences(code: str):
    """Stream (native_split, sentence) over the pooled train/dev/test files."""
    for split in ("train", "dev", "test"):
        with open(UD_DIR / f"{code}-ud-{split}.conllu", encoding="utf-8") as f:
            yield from ((split, sent) for sent in conllu.parse_incr(f))


def build_base_inventory(code: str):
    """Eligible surface tokens of one treebank, with exclusion counts."""
    records = []
    excl = {"mwt_range_token": 0, "empty_node": 0, "mwt_component_word": 0, "missing_lemma": 0}
    n_sent, seen_sids = 0, set()
    for split, sent in iter_pooled_sentences(code):
        n_sent += 1
        sid = sent.metadata["sent_id"]
        if sid in seen_sids:   # ids must stay unique after pooling
            raise RuntimeError(f"{code}: duplicate sent_id {sid!r} after pooling native splits")
        seen_sids.add(sid)

        mwt_members = set()   # ids covered by MWT range token
        for tok in sent:
            if isinstance(tok["id"], tuple) and tok["id"][1] == "-":
                mwt_members.update(range(tok["id"][0], tok["id"][2] + 1))

        for tok in sent:
            tid = tok["id"]
            if isinstance(tid, tuple):
                excl["mwt_range_token" if tid[1] == "-" else "empty_node"] += 1
                continue
            if tid in mwt_members:
                excl["mwt_component_word"] += 1
                continue
            if tok["lemma"] in (None, "_", ""):
                excl["missing_lemma"] += 1
                continue
            feats = tok["feats"] or {}
            records.append({
                "treebank": code, "native_split": split, "sent_id": sid, "word_id": tid,
                "form": tok["form"], "lemma": tok["lemma"], "upos": tok["upos"],
                "number": feats.get("Number"), "char_length": len(tok["form"]),
            })
    return pd.DataFrame(records), {"sentences": n_sent, "excluded": excl}

In [10]:
INVENTORY_DIR = CACHE_DIR / "inventory"
INVENTORY_DIR.mkdir(exist_ok=True)

inventories = {}   # (code, task) -> DataFrame
report = {}

for repo, code in CONFIG["treebanks"].items():
    base, stats = build_base_inventory(code)

    # UPOS: every eligible token with a gold tag
    has_upos = ~base["upos"].isin([None, "_"])
    inv_upos = base[has_upos].copy()
    inv_upos["gold_label"] = inv_upos["upos"]

    # Number: NOUN with Number in {Sing, Plur}
    is_noun = base["upos"] == "NOUN"
    noun_ok = is_noun & base["number"].isin(CONFIG["tasks"]["number"]["classes"])
    inv_num = base[noun_ok].copy()
    inv_num["gold_label"] = inv_num["number"]

    inventories[(code, "upos")] = inv_upos
    inventories[(code, "number")] = inv_num
    inv_upos.to_parquet(INVENTORY_DIR / f"{code}_upos.parquet", index=False)
    inv_num.to_parquet(INVENTORY_DIR / f"{code}_number.parquet", index=False)

    report[code] = {
        **stats,
        "eligible_tokens": len(base),
        "upos": {"n_targets": len(inv_upos),
                 "excluded_missing_upos": int((~has_upos).sum()),
                 "per_class_support": inv_upos["gold_label"].value_counts().to_dict()},
        "number": {"n_targets": len(inv_num),
                   "nouns_total": int(is_noun.sum()),
                   "nouns_excluded_no_or_other_number": int((is_noun & ~noun_ok).sum()),
                   "per_class_support": inv_num["gold_label"].value_counts().to_dict(),
                   "distinct_lemmas": int(inv_num["lemma"].nunique())},
    }
    print(f"{repo}: {stats['sentences']:,} pooled sents, {len(base):,} eligible tokens "
          f"(excluded: {stats['excluded']})")
    print(f"  UPOS   targets {len(inv_upos):>9,}   Number targets {len(inv_num):>7,} "
          f"(Sing/Plur = {report[code]['number']['per_class_support']})")

(RESULTS_DIR / "inventory_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\nwrote cache/inventory/*.parquet and results/inventory_report.json")

UD_English-EWT: 16,622 pooled sents, 247,988 eligible tokens (excluded: {'mwt_range_token': 3327, 'empty_node': 43, 'mwt_component_word': 6656, 'missing_lemma': 176})
  UPOS   targets   247,988   Number targets  42,552 (Sing/Plur = {'Sing': 32474, 'Plur': 10078})
UD_Italian-ISDT: 14,167 pooled sents, 258,533 eligible tokens (excluded: {'mwt_range_token': 19888, 'empty_node': 37, 'mwt_component_word': 39802, 'missing_lemma': 3})
  UPOS   targets   258,533   Number targets  54,926 (Sing/Plur = {'Sing': 38261, 'Plur': 16665})
UD_Polish-PDB: 22,152 pooled sents, 344,824 eligible tokens (excluded: {'mwt_range_token': 2495, 'empty_node': 0, 'mwt_component_word': 5154, 'missing_lemma': 0})
  UPOS   targets   344,824   Number targets  85,726 (Sing/Plur = {'Sing': 63014, 'Plur': 22712})

wrote cache/inventory/*.parquet and results/inventory_report.json


In [11]:
support = pd.DataFrame({code: inventories[(code, "upos")]["gold_label"].value_counts()
                        for code in CONFIG["treebanks"].values()}).fillna(0).astype(int)
support.index.name = "UPOS"
support = support.sort_values("en_ewt", ascending=False)
print(support.to_string())

       en_ewt  it_isdt  pl_pdb
UPOS                          
NOUN    42804    59478   88617
PUNCT   29762    33875   57873
VERB    27791    24210   37304
ADP     21769    26462   37155
PRON    21635    10201   16470
DET     20094    29845    9347
ADJ     16788    19775   35926
PROPN   15996    14776   11956
AUX     13541    11664    5948
ADV     12562    11421   11434
CCONJ    8203     8140   10446
PART     5194       26   10946
NUM      5049     5188    2633
SCONJ    4600     3028    7449
SYM       940      102      21
INTJ      931       65     177
X         329      277    1122


## 4. Target Alignment and Representation Extraction

Rebuilds each sentence's text from `FORM`, aligns every eligible target to its subwords, and extracts layer-8 last-subword vectors in one batched pass per treebank. Writes `cache/representations/{code}_layer8.npy`, a metadata parquet keyed by `(sent_id, word_id)`, and a marker recording the model revision and layer. Counts and subword distribution go to `results/extraction_report.json`.

In [12]:
def reconstruct_sentence(sent):
    """Rebuild the sentence text from FORM and SpaceAfter=No; return (text, spans)."""
    parts, spans = [], {}
    pos, skip_until = 0, 0
    for tok in sent:
        tid = tok["id"]
        if isinstance(tid, tuple):
            if tid[1] == ".": # empty node
                continue
            form = tok["form"] # MWT range token: surface form, once
            parts.append(form); pos += len(form)
            skip_until = tid[2]
            if (tok["misc"] or {}).get("SpaceAfter") != "No":
                parts.append(" "); pos += 1
            continue
        if tid <= skip_until: # component word of MWT range
            continue
        form = tok["form"]
        spans[tid] = (pos, pos + len(form))
        parts.append(form); pos += len(form)
        if (tok["misc"] or {}).get("SpaceAfter") != "No":
            parts.append(" "); pos += 1
    return "".join(parts).rstrip(), spans


# check: MWT contraction + SpaceAfter=No
_toy = conllu.parse("""# sent_id = toy-1
# text = Vado nel bosco.
1\tVado\tandare\tVERB\t_\t_\t0\troot\t_\t_
2-3\tnel\t_\t_\t_\t_\t_\t_\t_\t_
2\tin\tin\tADP\t_\t_\t4\tcase\t_\t_
3\til\til\tDET\t_\t_\t4\tdet\t_\t_
4\tbosco\tbosco\tNOUN\t_\t_\t1\tobl\t_\tSpaceAfter=No
5\t.\t.\tPUNCT\t_\t_\t1\tpunct\t_\t_
""")[0]
_text, _spans = reconstruct_sentence(_toy)
assert _text == _toy.metadata["text"] == "Vado nel bosco.", _text
assert _spans == {1: (0, 4), 4: (9, 14), 5: (14, 15)} and 2 not in _spans and 3 not in _spans
assert _text[slice(*_spans[4])] == "bosco"
print(f"reconstruction ✓  {_text!r}  spans={_spans} (components 2,3 correctly span-less)")

reconstruction ✓  'Vado nel bosco.'  spans={1: (0, 4), 4: (9, 14), 5: (14, 15)} (components 2,3 correctly span-less)


In [13]:
# extraction loop, cached per treebank
import gc
import time


def pack_batches(idx_sorted_desc, lengths, batch_slots=8192, max_batch=256):
    """Greedy packing of length-sorted sentences; bounds transient memory only."""
    batches, cur = [], []
    for i in idx_sorted_desc:
        if cur and (len(cur) + 1 > max_batch or (len(cur) + 1) * lengths[cur[0]] > batch_slots):
            batches.append(cur)
            cur = []
        cur.append(i)
    if cur:
        batches.append(cur)
    return batches


def extract_treebank(code: str, need_vectors: bool = True):
    """Extract and cache the frozen layer-8 vectors of `code`.
    With need_vectors=False, a copy that holds the extraction record but not the
    vectors is left as it is; load_vectors() extracts them when a probe needs them."""
    marker_p = REPR_DIR / f"{code}.marker.json"
    npy_p = REPR_DIR / f"{code}_layer8.npy"
    meta_p = REPR_DIR / f"{code}_meta.parquet"

    if marker_p.exists():
        info = json.loads(marker_p.read_text(encoding="utf-8"))
        if (info["model_revision"], info["layer_index"]) != (MODEL_REVISION, CONFIG["layer_index"]):
            raise RuntimeError( # never mix representations
                f"{code}: cache at {marker_p} was built with different model/layer "
                f"({info['model_revision'][:12]}/L{info['layer_index']}) — delete "
                f"cache/representations to re-extract under the current config")
        if npy_p.exists():
            print(f"{code}: cache hit — reusing {info['n_rows']:,} frozen vectors")
            return info
        if not need_vectors:
            print(f"{code}: extraction record found, vectors not in this copy "
                  f"(extracted only if a probe has to be fitted)")
            return info

    # union of both task target sets
    cols = ["treebank", "native_split", "sent_id", "word_id", "form", "lemma", "upos",
            "number", "char_length"]
    union = (pd.concat([inventories[(code, "upos")][cols], inventories[(code, "number")][cols]])
             .drop_duplicates(["sent_id", "word_id"]))
    by_key = {(r.sent_id, r.word_id): r for r in union.itertuples(index=False)}
    wids_by_sid = union.groupby("sent_id")["word_id"].agg(list).to_dict()

    # pass 1: text + char spans, cross-checked against '# text'
    sents, text_mismatch = [], 0
    for _, sent in iter_pooled_sentences(code):
        sid = sent.metadata["sent_id"]
        wids = wids_by_sid.get(sid)
        if not wids:
            continue
        text, spans = reconstruct_sentence(sent)
        if sent.metadata.get("text") not in (None, text):
            text_mismatch += 1
        sents.append((sid, text, [(w, *spans[w]) for w in wids]))

    # pass 2: subword lengths for batch packing
    texts = [t for _, t, _ in sents]
    lens = [len(ids) for ids in tokenizer(texts, add_special_tokens=True, truncation=True,
                                          max_length=CONFIG["max_length"])["input_ids"]]
    batches = pack_batches(sorted(range(len(sents)), key=lambda i: -lens[i]), lens)

    # pass 3: batched inference, last-subword readout
    X = np.empty((len(union), 768), dtype=np.float32)
    meta_rows, dropped = [], []
    row, t0 = 0, time.time()
    for bi, batch in enumerate(batches):
        enc = tokenizer([texts[i] for i in batch], padding=True, truncation=True,
                        max_length=CONFIG["max_length"], add_special_tokens=True,
                        return_offsets_mapping=True, return_tensors="pt")
        with torch.no_grad():
            out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        layer = out.hidden_states[CONFIG["layer_index"]]
        for j, i_sent in enumerate(batch):
            sid, _, tgts = sents[i_sent]
            offs = enc["offset_mapping"][j].tolist() # padding rows are (0,0): never match
            for wid, s, e in tgts:
                idxs = subwords_for_span(offs, s, e)
                r = by_key[(sid, wid)]
                if not idxs:   # dropped and logged
                    dropped.append({"sent_id": sid, "word_id": wid, "form": r.form})
                    continue
                X[row] = layer[j, idxs[-1]].numpy()
                meta_rows.append({"treebank": code, "native_split": r.native_split,
                                  "sent_id": sid, "word_id": wid, "form": r.form,
                                  "lemma": r.lemma, "upos": r.upos, "number": r.number,
                                  "char_length": r.char_length, "n_subwords": len(idxs),
                                  "row": row})
                row += 1
        if (bi + 1) % 25 == 0:
            print(f"  {code}: batch {bi + 1}/{len(batches)}  ({time.time() - t0:,.0f}s)")
        del enc, out, layer

    np.save(npy_p, X[:row])
    pd.DataFrame(meta_rows).to_parquet(meta_p, index=False)
    info = {"model_revision": MODEL_REVISION, "layer_index": CONFIG["layer_index"],
            "n_rows": row, "n_targets": len(union), "n_sentences": len(sents),
            "dropped_zero_subword": len(dropped), "dropped_examples": dropped[:5],
            "text_metadata_mismatches": text_mismatch,
            "elapsed_s": round(time.time() - t0, 1)}
    marker_p.write_text(json.dumps(info, indent=2), encoding="utf-8")
    del X
    gc.collect()
    print(f"{code}: {row:,}/{len(union):,} vectors from {len(sents):,} sentences "
          f"in {info['elapsed_s']:,.0f}s (dropped {len(dropped)}, "
          f"#text mismatches {text_mismatch}) -> {npy_p.name}")
    return info


def load_vectors(code: str):
    """The frozen layer-8 vectors of `code`, memory-mapped; extracted first if this copy lacks them."""
    npy_p = REPR_DIR / f"{code}_layer8.npy"
    if not npy_p.exists():
        extract_treebank(code)
    return np.load(npy_p, mmap_mode="r")


extraction_report = {code: extract_treebank(code, need_vectors=False)
                     for code in CONFIG["treebanks"].values()}
(RESULTS_DIR / "extraction_report.json").write_text(
    json.dumps(extraction_report, indent=2), encoding="utf-8")
print("\nwrote results/extraction_report.json")

en_ewt: cache hit — reusing 247,988 frozen vectors
it_isdt: cache hit — reusing 258,533 frozen vectors
pl_pdb: cache hit — reusing 344,824 frozen vectors

wrote results/extraction_report.json


In [14]:
# cache integrity + batched-vs-single consistency
import hashlib

N_CHECK_SENTENCES = 50 # sampled per treebank
TOL = 1e-4 # batched and single GEMMs accumulate in a different order


def check_seed(code: str) -> int:
    """Per-treebank stream for the consistency sample, derived from the global seed."""
    return int.from_bytes(hashlib.sha256(f"{SEED}:consistency:{code}".encode()).digest()[:8], "big")


frag_stats = {}
for code in CONFIG["treebanks"].values():
    npy_p = REPR_DIR / f"{code}_layer8.npy"
    X = np.load(npy_p, mmap_mode="r") if npy_p.exists() else None   # not in the repository
    meta = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet")

    assert len(meta) == extraction_report[code]["n_rows"]
    assert X is None or (X.shape == (len(meta), 768) and X.dtype == np.float32)
    assert (meta["row"].to_numpy() == np.arange(len(meta))).all()
    assert not meta.duplicated(["sent_id", "word_id"]).any()

    # every target is cached or among the logged drops
    cached = pd.MultiIndex.from_frame(meta[["sent_id", "word_id"]])
    for task in ("upos", "number"):
        inv = inventories[(code, task)]
        n_missing = (~pd.MultiIndex.from_frame(inv[["sent_id", "word_id"]]).isin(cached)).sum()
        assert n_missing <= extraction_report[code]["dropped_zero_subword"], (code, task, n_missing)

    # n_subwords summary for extraction report
    nsw = meta["n_subwords"]
    frag_stats[code] = {"mean": float(nsw.mean()),
                        "pct_single_subword": float(100.0 * (nsw == 1).mean()),
                        "p95": float(nsw.quantile(0.95)),
                        "max": int(nsw.max()),
                        "max_form": str(meta.loc[nsw.idxmax(), "form"])}

    if X is None:
        print(f"consistency skipped for {code}: no cached vectors in this copy to compare against")
        continue

    # right-padding must not perturb a target's vector, or the readout would depend on its batch
    key2row = {(s, w): (r, n) for s, w, r, n in
               zip(meta["sent_id"], meta["word_id"], meta["row"], meta["n_subwords"])}
    sids = meta["sent_id"].unique()
    rng = np.random.default_rng(check_seed(code))
    sample = set(rng.choice(sids, size=min(N_CHECK_SENTENCES, len(sids)), replace=False).tolist())

    n_sent, n_tgt, max_dev = 0, 0, 0.0
    for _, sent in iter_pooled_sentences(code):
        sid = sent.metadata["sent_id"]
        if sid not in sample:
            continue
        text, spans = reconstruct_sentence(sent)
        offs, layer = encode_sentence(text)
        for w, (s, e) in spans.items():
            if (sid, w) not in key2row:
                continue
            vec, n_sub = last_subword_vector(offs, layer, s, e)
            r, n_cached = key2row[(sid, w)]
            assert n_sub == n_cached, (code, sid, w)
            dev = float(np.abs(X[r] - vec.numpy()).max())
            assert dev <= TOL, (code, sid, w, dev)
            max_dev = max(max_dev, dev)
            n_tgt += 1
        n_sent += 1
    print(f"consistency \u2713  {code}: {n_tgt:,} targets over {n_sent} sampled sentences "
          f"match the single-sentence path (max |\u0394| = {max_dev:.2e}, n_subwords exact)")

for code, stats in frag_stats.items():
    extraction_report[code]["n_subwords"] = stats
(RESULTS_DIR / "extraction_report.json").write_text(
    json.dumps(extraction_report, indent=2), encoding="utf-8")

print("fragmentation (n_subwords) -> results/extraction_report.json")
print(pd.DataFrame(frag_stats).T.drop(columns=["max_form"]).to_string())


consistency ✓  en_ewt: 986 targets over 50 sampled sentences match the single-sentence path (max |Δ| = 2.28e-05, n_subwords exact)
consistency ✓  it_isdt: 931 targets over 50 sampled sentences match the single-sentence path (max |Δ| = 1.72e-05, n_subwords exact)
consistency ✓  pl_pdb: 905 targets over 50 sampled sentences match the single-sentence path (max |Δ| = 1.53e-05, n_subwords exact)
fragmentation (n_subwords) -> results/extraction_report.json
             mean pct_single_subword  p95  max
en_ewt   1.212909          85.541639  2.0  305
it_isdt  1.327567          74.934341  3.0   24
pl_pdb   1.600048          63.155697  4.0   17


## 5. Paired Split Construction and Feasibility Gate

Draws the split once per (language, task) from the cached metadata, with parameters and thresholds fixed in `config.json`, and gates every cell before any probe is trained. Writes `cache/splits/` and `results/split_report.json`; the notebook reuses the frozen files there, checks them against the checksums in `cache/splits/SPLITS.json` and against the current inventory, and stops rather than redrawing the different, equally valid split a rebuild would give.

In [15]:
import hashlib

SPLIT_CFG, FEAS = CONFIG["split"], CONFIG["feasibility"]
SPLIT_DIR.mkdir(exist_ok=True)
SPLIT_MANIFEST = SPLIT_DIR / "SPLITS.json"
split_checksums = (json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))["files"]
                   if SPLIT_MANIFEST.exists() else {})


def cell_seed(code, task):
    """Stable per-cell seed derived from the global SEED (hashlib -> reproducible across runs)."""
    return int.from_bytes(hashlib.sha256(f"{SEED}:{code}:{task}".encode()).digest()[:8], "big")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def build_paired_split(code, task):
    """Tight size-matched split for one (language, task); returns (occurrences, summary)."""
    inv = inventories[(code, task)][["sent_id", "word_id", "lemma", "gold_label"]]
    meta = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet", columns=["sent_id", "word_id", "row"])
    occ = (inv.merge(meta, on=["sent_id", "word_id"]).reset_index(drop=True)
           .rename(columns={"row": "repr_row"}))
    assert len(occ) == len(inv), f"{code}/{task}: {len(inv) - len(occ)} occurrences without a representation"

    rng = np.random.default_rng(cell_seed(code, task))
    N = len(occ)
    lemma_arr, gold = occ["lemma"].to_numpy(), occ["gold_label"].to_numpy()
    by_lemma = occ.groupby("lemma").indices   # lemma -> positional indices
    counts = {lem: len(pos) for lem, pos in by_lemma.items()}

    # H: multi-occurrence lemmata until the projected |T| reaches 20%
    cand = np.array([l for l, c in counts.items() if c >= 2], dtype=object)
    rng.shuffle(cand)
    target_T = round(SPLIT_CFG["test_fraction_target"] * N)
    floor_H = min(SPLIT_CFG["min_heldout_lemmas"], len(cand))
    H, proj = set(), 0
    for lem in cand:
        if proj >= target_T and len(H) >= floor_H:
            break
        H.add(lem)
        proj += counts[lem] // 2

    # Number: extend H until both classes are reachable in T
    if task == "number":
        pool_cls = set(gold[np.concatenate([by_lemma[l] for l in H])].tolist())
        for cls in [c for c in CONFIG["tasks"]["number"]["classes"] if c not in pool_cls]:
            for lem in cand:
                if lem not in H and cls in gold[by_lemma[lem]]:
                    H.add(lem)
                    break

    # 50/50 per held-out lemma, at least one each side
    part = np.empty(N, dtype=object)
    tf = SPLIT_CFG["per_lemma_test_seen"][0]
    for lem in sorted(H):
        pos = by_lemma[lem].copy()
        rng.shuffle(pos)
        nt = max(1, min(len(pos) - 1, int(len(pos) * tf)))
        part[pos[:nt]] = "test"
        part[pos[nt:]] = "seen"

    # P -> reserve R (|R| = s) and core K
    bg = np.concatenate([by_lemma[l] for l in counts if l not in H])
    s = int((part == "seen").sum())
    assert len(bg) >= s, f"{code}/{task}: |P|={len(bg)} < s={s} — size-match precondition impossible"
    bg = bg[rng.permutation(len(bg))]
    part[bg[:s]] = "reserve"
    part[bg[s:]] = "core"
    occ["partition"] = part
    assert sum(int((part == k).sum()) for k in ("test", "seen", "core", "reserve")) == N

    trainA, trainB, is_test = np.isin(part, ["core", "seen"]), np.isin(part, ["core", "reserve"]), part == "test"
    cc = lambda m: {k: int(v) for k, v in zip(*np.unique(gold[m], return_counts=True))}
    summary = {
        "N": N, "target_T": target_T, "n_H_lemmas": len(H), "n_B_lemmas": len(counts) - len(H),
        "T": int(is_test.sum()), "S": s, "P": int(len(bg)),
        "K": int((part == "core").sum()), "R": int((part == "reserve").sum()),
        "T_fraction": round(float(is_test.mean()), 4),
        "train_A": int(trainA.sum()), "train_B": int(trainB.sum()),
        "T_by_class": cc(is_test), "train_A_by_class": cc(trainA), "train_B_by_class": cc(trainB),
        "contributing_H_lemmas": int(np.unique(lemma_arr[is_test]).size),
        "trainB_lemmas_in_H": int(len(set(np.unique(lemma_arr[trainB]).tolist()) & H)),
    }
    return occ[["sent_id", "word_id", "repr_row", "lemma", "gold_label", "partition"]], summary


def check_against_inventory(code, task, occ):
    """A reused split must still address the current targets, labels and metadata rows."""
    inv = inventories[(code, task)][["sent_id", "word_id", "lemma", "gold_label"]]
    meta = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet", columns=["sent_id", "word_id", "row"])
    m = occ.merge(inv, on=["sent_id", "word_id"], suffixes=("", "_now")).merge(meta, on=["sent_id", "word_id"])
    assert len(m) == len(occ) == len(inv), (
        f"{code}/{task}: the cached split has {len(occ):,} occurrences and matches {len(m):,} of the "
        f"{len(inv):,} current targets")
    assert (m["gold_label"] == m["gold_label_now"]).all(), f"{code}/{task}: gold labels have changed"
    assert (m["lemma"] == m["lemma_now"]).all(), f"{code}/{task}: lemmata have changed"
    assert (m["repr_row"] == m["row"]).all(), f"{code}/{task}: repr_row no longer addresses the metadata"


def get_paired_split(code, task):
    """Build once and freeze; reuse the cached split when params and checksums match (raise otherwise)."""
    pq, mk = SPLIT_DIR / f"{code}_{task}.parquet", SPLIT_DIR / f"{code}_{task}.split.json"
    params = {"seed": cell_seed(code, task),
              **{k: SPLIT_CFG[k] for k in
                 ("test_fraction_target", "min_heldout_lemmas", "per_lemma_test_seen", "size_matching")}}
    missing = [p.name for p in (pq, mk) if not p.exists()]
    if split_checksums and missing:
        raise RuntimeError(f"{code}/{task}: cache/splits/{SPLIT_MANIFEST.name} records this split, but "
                           f"cache/splits is missing {', '.join(missing)} — restore it from the repository; "
                           f"a rebuild draws a different split, and the thesis numbers come from these files")
    if mk.exists():
        info = json.loads(mk.read_text(encoding="utf-8"))
        if info["params"] != params:
            raise RuntimeError(f"{code}/{task}: cached split built with different params — "
                               f"delete cache/splits to rebuild under the current config")
        for p in (pq, mk):
            if p.name in split_checksums and sha256(p) != split_checksums[p.name]:
                raise RuntimeError(f"{p.name}: sha256 does not match cache/splits/{SPLIT_MANIFEST.name} — "
                                   f"restore the file from the repository")
        occ = pd.read_parquet(pq)
        check_against_inventory(code, task, occ)
        return occ, info["summary"]
    occ, summary = build_paired_split(code, task)
    occ.to_parquet(pq, index=False)
    mk.write_text(json.dumps({"params": params, "summary": summary}, indent=2), encoding="utf-8")
    return occ, summary


splits, split_summaries = {}, {}
for code in CONFIG["treebanks"].values():
    for task in ("upos", "number"):
        occ, summ = get_paired_split(code, task)
        splits[(code, task)], split_summaries[(code, task)] = occ, summ
        print(f"{code:8s} {task:6s}: N={summ['N']:>7,}  |H|={summ['n_H_lemmas']:>5,}  "
              f"|T|={summ['T']:>7,} ({summ['T_fraction']:.0%})  |P|={summ['P']:>7,}  "
              f"s={summ['S']:>6,}  train_A=train_B={summ['train_A']:>7,}")

if not SPLIT_MANIFEST.exists():
    SPLIT_MANIFEST.write_text(json.dumps({"files": {
        p.name: sha256(p) for code in CONFIG["treebanks"].values() for task in ("upos", "number")
        for p in (SPLIT_DIR / f"{code}_{task}.parquet", SPLIT_DIR / f"{code}_{task}.split.json")}},
        indent=2), encoding="utf-8")

(RESULTS_DIR / "split_report.json").write_text(
    json.dumps({f"{c}/{t}": s for (c, t), s in split_summaries.items()}, indent=2), encoding="utf-8")
print(f"\nsplits verified against cache/splits/{SPLIT_MANIFEST.name} ({len(split_checksums)} checksums) "
      f"and the current inventory" if split_checksums else
      f"\nwrote cache/splits/*.parquet and cache/splits/{SPLIT_MANIFEST.name}")
print("wrote results/split_report.json")

en_ewt   upos  : N=247,988  |H|=4,785  |T|= 49,630 (20%)  |P|=146,817  s=51,541  train_A=train_B=146,817
en_ewt   number: N= 42,552  |H|=1,696  |T|=  8,510 (20%)  |P|= 24,841  s= 9,201  train_A=train_B= 24,841
it_isdt  upos  : N=258,533  |H|=3,287  |T|= 51,709 (20%)  |P|=153,848  s=52,976  train_A=train_B=153,848
it_isdt  number: N= 54,926  |H|=1,636  |T|= 10,989 (20%)  |P|= 32,289  s=11,648  train_A=train_B= 32,289
pl_pdb   upos  : N=344,824  |H|=6,751  |T|= 72,841 (21%)  |P|=196,499  s=75,484  train_A=train_B=196,499
pl_pdb   number: N= 85,726  |H|=2,639  |T|= 17,155 (20%)  |P|= 50,373  s=18,198  train_A=train_B= 50,373

splits verified against cache/splits/SPLITS.json (12 checksums) and the current inventory
wrote results/split_report.json


In [16]:
# feasibility gate
def gate_fails(s, task):
    """Threshold checks per (language, task); returns the list of violated thresholds."""
    f = []
    if s["n_H_lemmas"] < FEAS["min_heldout_lemmas"]:
        f.append(f"|H|={s['n_H_lemmas']}<{FEAS['min_heldout_lemmas']}")
    if s["T"] < FEAS["min_test_instances"]:
        f.append(f"|T|={s['T']}<{FEAS['min_test_instances']}")
    if s["contributing_H_lemmas"] < FEAS["min_contributing_heldout_lemmas"]:
        f.append(f"contrib={s['contributing_H_lemmas']}<{FEAS['min_contributing_heldout_lemmas']}")
    if s["P"] < s["S"]:
        f.append(f"|P|={s['P']}<s={s['S']}")
    if task == "number":
        for cls in CONFIG["tasks"]["number"]["classes"]:
            if s["T_by_class"].get(cls, 0) < FEAS["min_per_class_number"]:
                f.append(f"T[{cls}]={s['T_by_class'].get(cls, 0)}<{FEAS['min_per_class_number']}")
            for cond in ("A", "B"):
                if s[f"train_{cond}_by_class"].get(cls, 0) < 1:
                    f.append(f"train_{cond}[{cls}]=0")
    return f


rows, failures = [], {}
for (code, task), s in split_summaries.items():
    fails = gate_fails(s, task)
    if fails:
        failures[f"{code}/{task}"] = fails
    for cond in ("A", "B"):
        tb = s[f"train_{cond}_by_class"]
        rows.append({
            "language": code, "task": task, "cond": cond,
            "H_lem": s["n_H_lemmas"], "B_lem": s["n_B_lemmas"], "contrib_H": s["contributing_H_lemmas"],
            "|T|": s["T"], "T_frac": f"{s['T_fraction']:.0%}",
            "T_Sing": s["T_by_class"].get("Sing", ""), "T_Plur": s["T_by_class"].get("Plur", ""),
            "|P|": s["P"], "s": s["S"], "sizematch": s["P"] >= s["S"], "|train|": s[f"train_{cond}"],
            "tr_Sing": tb.get("Sing", ""), "tr_Plur": tb.get("Plur", ""),
            "PASS": not fails,
        })

feas_report = {
    "thresholds": FEAS,
    "all_pass": not failures,
    "cells": {f"{code}/{task}": {
        "n_H_lemmas": s["n_H_lemmas"], "n_B_lemmas": s["n_B_lemmas"], "T": s["T"],
        "T_by_class": s["T_by_class"], "P": s["P"], "s": s["S"],
        "train_A_by_class": s["train_A_by_class"], "train_B_by_class": s["train_B_by_class"],
        "contributing_H_lemmas": s["contributing_H_lemmas"],
        "fails": failures.get(f"{code}/{task}", []),
    } for (code, task), s in split_summaries.items()},
}
(RESULTS_DIR / "feasibility_report.json").write_text(json.dumps(feas_report, indent=2), encoding="utf-8")

print(pd.DataFrame(rows).to_string(index=False))
print()
if failures:
    raise RuntimeError(f"feasibility gate failed: {failures}")
print("all 6 (language × task) cells pass the feasibility gate")

language   task cond  H_lem  B_lem  contrib_H   |T| T_frac T_Sing T_Plur    |P|     s  sizematch  |train| tr_Sing tr_Plur  PASS
  en_ewt   upos    A   4785  11741       4785 49630    20%               146817 51541       True   146817                  True
  en_ewt   upos    B   4785  11741       4785 49630    20%               146817 51541       True   146817                  True
  en_ewt number    A   1696   4340       1696  8510    20%   6477   2033  24841  9201       True    24841   18960    5881  True
  en_ewt number    B   1696   4340       1696  8510    20%   6477   2033  24841  9201       True    24841   19030    5811  True
 it_isdt   upos    A   3287  16163       3287 51709    20%               153848 52976       True   153848                  True
 it_isdt   upos    B   3287  16163       3287 51709    20%               153848 52976       True   153848                  True
 it_isdt number    A   1636   4725       1636 10989    20%   7517   3472  32289 11648       True    3228

In [17]:
# formal properties, from the frozen split files
for (code, task) in split_summaries:
    d = pd.read_parquet(SPLIT_DIR / f"{code}_{task}.parquet")
    part, lem = d["partition"].to_numpy(), d["lemma"].to_numpy()
    trainA, trainB, T = np.isin(part, ["core", "seen"]), np.isin(part, ["core", "reserve"]), part == "test"
    H = set(lem[np.isin(part, ["test", "seen"])].tolist())
    B = set(lem[np.isin(part, ["core", "reserve"])].tolist())

    assert not (H & B), f"{code}/{task}: a lemma is both held-out and background"
    assert not (set(lem[trainB].tolist()) & H), f"{code}/{task}: train_B not lemma-disjoint from H"
    assert H <= set(lem[trainA].tolist()), f"{code}/{task}: some H lemma missing from train_A"
    nP = int(trainB.sum())
    assert int(trainA.sum()) == int(trainB.sum()) == nP, f"{code}/{task}: training sizes not matched"
    assert (part == "seen").sum() == (part == "reserve").sum(), f"{code}/{task}: |S| != |R|"
    assert not (T & trainA).any() and not (T & trainB).any(), f"{code}/{task}: T leaks into training"
    n_vec = extraction_report[code]["n_rows"]   # rows of the vectors, present in this copy or not
    assert d["repr_row"].is_unique and d["repr_row"].min() >= 0 and d["repr_row"].max() < n_vec

print("formal properties of the split verified for all 6 cells:")
print("  • H / B lemma partition is clean; condition-B training is lemma-disjoint from H")
print("  • condition-A training contains every held-out lemma (via Seen)")
print("  • |train_A| = |train_B| = |P| and |S| = |R| (tight size match)")
print("  • the paired test set T is held out of both conditions; repr rows valid & unique")

formal properties of the split verified for all 6 cells:
  • H / B lemma partition is clean; condition-B training is lemma-disjoint from H
  • condition-A training contains every held-out lemma (via Seen)
  • |train_A| = |train_B| = |P| and |S| = |R| (tight size match)
  • the paired test set T is held out of both conditions; repr rows valid & unique


## 6. Standardization and Probe Training

Fits the scaler and the probe per condition on the frozen split and caches the predictions on T, keyed to the split, model revision, and probe settings. Writes `cache/predictions/` and `results/main_probe_report.json`.

In [18]:
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning

# the one probe spec, identical across A / B / control
PROBE_KW = dict(penalty="l2", C=CONFIG["probe"]["C"], solver=CONFIG["probe"]["solver"],
                max_iter=CONFIG["probe"]["max_iter"], class_weight=CONFIG["probe"]["class_weight"],
                random_state=SEED)


def fit_probe(X_train, y_train):
    """Fit the probe on one condition; return (pipeline, n_iter, converged)."""
    pipe = make_pipeline(StandardScaler(), LogisticRegression(**PROBE_KW))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        pipe.fit(X_train, y_train)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return pipe, int(np.max(pipe[-1].n_iter_)), converged


def point_metrics(y_true, y_pred):
    """Accuracy and macro-F1 (unweighted mean over classes; zero_division=0 for absent classes)."""
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0))}


print("probe spec:", PROBE_KW)

probe spec: {'penalty': 'l2', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 2000, 'class_weight': None, 'random_state': 42}


In [19]:
# train probes per cell and condition; cache predictions on T
def majority_test_baseline(y_test):
    """The common floor: T's own most frequent label everywhere."""
    lab = str(pd.Series(y_test).value_counts().idxmax())
    return {"label": lab, **point_metrics(y_test, np.full(len(y_test), lab))}


main_report = {}
for code in CONFIG["treebanks"].values():
    X = None   # loaded only if a probe has to be fitted
    for task in ("upos", "number"):
        pred_p = PRED_DIR / f"{code}_{task}.parquet"
        mk = PRED_DIR / f"{code}_{task}.pred.json"
        params = {"model_revision": MODEL_REVISION, "layer_index": CONFIG["layer_index"],
                  "split_seed": cell_seed(code, task), "probe": PROBE_KW}
        if mk.exists():
            info = json.loads(mk.read_text(encoding="utf-8"))
            if info["params"] == params:
                main_report[(code, task)] = info["summary"]
                print(f"{code:8s} {task:6s}: cache hit ({info['summary']['T']:,} test predictions)")
                continue
            raise RuntimeError(f"{code}/{task}: cached predictions built with different params — "
                               f"delete cache/predictions to retrain under the current config")

        if X is None:
            X = load_vectors(code)
        d = splits[(code, task)]
        repr_row = d["repr_row"].to_numpy()
        y = d["gold_label"].to_numpy()
        part = d["partition"].to_numpy()
        test_df = d[part == "test"].sort_values("repr_row").reset_index(drop=True)
        Xte = np.asarray(X[test_df["repr_row"].to_numpy()])
        yte = test_df["gold_label"].to_numpy()

        summary = {"task": task, "T": len(test_df), "conditions": {}}
        for cond, keep in (("A", ["core", "seen"]), ("B", ["core", "reserve"])):
            mask = np.isin(part, keep)
            pipe, n_iter, converged = fit_probe(np.asarray(X[repr_row[mask]]), y[mask])
            pred = pipe.predict(Xte)
            test_df[f"pred_{cond}"] = pred
            summary["conditions"][cond] = {
                "n_train": int(mask.sum()), "n_iter": n_iter, "converged": converged,
                "probe": point_metrics(yte, pred),
            }
            del pipe
        # floor depends on T alone
        summary["majority_test"] = majority_test_baseline(yte)

        test_df.to_parquet(pred_p, index=False)
        mk.write_text(json.dumps({"params": params, "summary": summary}, indent=2), encoding="utf-8")
        main_report[(code, task)] = summary
        A, B = summary["conditions"]["A"], summary["conditions"]["B"]
        print(f"{code:8s} {task:6s}: acc A={A['probe']['accuracy']:.3f} B={B['probe']['accuracy']:.3f}"
              f"  (floor {summary['majority_test']['accuracy']:.3f})  n_iter A/B={A['n_iter']}/{B['n_iter']}"
              f"  converged={A['converged'] and B['converged']}")
    del X
    gc.collect()

(RESULTS_DIR / "main_probe_report.json").write_text(
    json.dumps({f"{c}/{t}": s for (c, t), s in main_report.items()}, indent=2), encoding="utf-8")
print("\nwrote cache/predictions/*.parquet and results/main_probe_report.json")

en_ewt   upos  : cache hit (49,630 test predictions)
en_ewt   number: cache hit (8,510 test predictions)
it_isdt  upos  : cache hit (51,709 test predictions)
it_isdt  number: cache hit (10,989 test predictions)
pl_pdb   upos  : cache hit (72,841 test predictions)
pl_pdb   number: cache hit (17,155 test predictions)

wrote cache/predictions/*.parquet and results/main_probe_report.json


In [20]:
# readout: probe against floor, condition A vs B
rows, any_nonconv = [], False
for (code, task), s in main_report.items():
    A, B = s["conditions"]["A"], s["conditions"]["B"]
    mt = s["majority_test"]
    any_nonconv |= not (A["converged"] and B["converged"])
    rows.append({
        "language": code, "task": task, "|T|": s["T"],
        "acc_A": A["probe"]["accuracy"], "acc_B": B["probe"]["accuracy"],
        "Δacc": A["probe"]["accuracy"] - B["probe"]["accuracy"],
        "f1_A": A["probe"]["macro_f1"], "f1_B": B["probe"]["macro_f1"],
        "Δf1": A["probe"]["macro_f1"] - B["probe"]["macro_f1"],
        "majT_acc": mt["accuracy"], "majT_f1": mt["macro_f1"],
    })
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
if any_nonconv:
    print("warning: a probe reached max_iter without converging; its predictions are used as they are.")
else:
    print(f"all 12 probes converged within max_iter = {CONFIG['probe']['max_iter']}.")

language   task   |T|  acc_A  acc_B  Δacc  f1_A  f1_B   Δf1  majT_acc  majT_f1
  en_ewt   upos 49630  0.959  0.866 0.093 0.919 0.677 0.242     0.210    0.020
  en_ewt number  8510  0.991  0.986 0.005 0.988 0.981 0.007     0.761    0.432
 it_isdt   upos 51709  0.977  0.839 0.138 0.860 0.683 0.177     0.203    0.021
 it_isdt number 10989  0.996  0.991 0.005 0.995 0.990 0.005     0.684    0.406
  pl_pdb   upos 72841  0.981  0.847 0.135 0.905 0.622 0.283     0.284    0.026
  pl_pdb number 17155  0.989  0.985 0.004 0.986 0.982 0.005     0.719    0.418

all 12 probes converged within max_iter = 2000.


## 7. Control Task, Selectivity, and Primary Evaluation

Draws one control label per lemma type, trains the same probe on the same split to predict it, and computes selectivity, the gap, its bootstrap interval, McNemar's test, and the Holm correction from the cached predictions. Writes `cache/control/`, `results/main_results.csv`, `gap_table.csv`, `mcnemar_table.csv`, and `selectivity_report.json`.

In [21]:
CONTROL_DIR = CACHE_DIR / "control"
CONTROL_DIR.mkdir(exist_ok=True)


def control_seed(code, task):
    """Stable per-cell seed for the control-label draw (independent of the split seed)."""
    return int.from_bytes(hashlib.sha256(f"{SEED}:control:{code}:{task}".encode()).digest()[:8], "big")


def assign_control_labels(code, task):
    """One random control label per lemma type, drawn from the label marginal. Cached."""
    p = CONTROL_DIR / f"{code}_{task}.parquet"
    if p.exists():
        m = pd.read_parquet(p)
        return dict(zip(m["lemma"], m["control_label"]))
    inv = inventories[(code, task)]
    labels, counts = np.unique(inv["gold_label"].to_numpy(), return_counts=True)
    marginal = counts / counts.sum()
    lemmas = pd.unique(inv["lemma"])
    drawn = np.random.default_rng(control_seed(code, task)).choice(labels, size=len(lemmas), p=marginal)
    pd.DataFrame({"lemma": lemmas, "control_label": drawn}).to_parquet(p, index=False)
    return dict(zip(lemmas, drawn))


control_maps = {(code, task): assign_control_labels(code, task)
                for code in CONFIG["treebanks"].values() for task in ("upos", "number")}
for (code, task), cm in control_maps.items():
    top = pd.Series(list(cm.values())).value_counts(normalize=True).round(2).head(3).to_dict()
    print(f"{code:8s} {task:6s}: {len(cm):>6,} lemma types assigned control labels  (top~{top})")
print("\nwrote cache/control/*.parquet")

en_ewt   upos  : 16,526 lemma types assigned control labels  (top~{'NOUN': 0.17, 'PUNCT': 0.12, 'VERB': 0.11})
en_ewt   number:  6,036 lemma types assigned control labels  (top~{'Sing': 0.76, 'Plur': 0.24})
it_isdt  upos  : 19,450 lemma types assigned control labels  (top~{'NOUN': 0.23, 'PUNCT': 0.13, 'DET': 0.12})
it_isdt  number:  6,361 lemma types assigned control labels  (top~{'Sing': 0.7, 'Plur': 0.3})
pl_pdb   upos  : 28,137 lemma types assigned control labels  (top~{'NOUN': 0.25, 'PUNCT': 0.17, 'ADP': 0.11})
pl_pdb   number: 11,061 lemma types assigned control labels  (top~{'Sing': 0.73, 'Plur': 0.27})

wrote cache/control/*.parquet


In [22]:
# control probes
control_report = {}
for code in CONFIG["treebanks"].values():
    X = None   # loaded only if a probe has to be fitted
    for task in ("upos", "number"):
        pred_p = PRED_DIR / f"{code}_{task}_control.parquet"
        mk = PRED_DIR / f"{code}_{task}_control.pred.json"
        params = {"model_revision": MODEL_REVISION, "layer_index": CONFIG["layer_index"],
                  "split_seed": cell_seed(code, task), "control_seed": control_seed(code, task),
                  "probe": PROBE_KW}
        if mk.exists():
            info = json.loads(mk.read_text(encoding="utf-8"))
            if info["params"] == params:
                control_report[(code, task)] = info["summary"]
                print(f"{code:8s} {task:6s}: control cache hit")
                continue
            raise RuntimeError(f"{code}/{task}: control cache params differ - delete cache/predictions")

        if X is None:
            X = load_vectors(code)
        d = splits[(code, task)]
        cmap = control_maps[(code, task)]
        repr_row, part = d["repr_row"].to_numpy(), d["partition"].to_numpy()
        yctrl = pd.Series(d["lemma"].to_numpy()).map(cmap).to_numpy()
        assert not pd.isna(yctrl).any(), f"{code}/{task}: lemma without a control label"
        test_df = d[part == "test"].sort_values("repr_row").reset_index(drop=True)
        Xte = np.asarray(X[test_df["repr_row"].to_numpy()])
        yte = pd.Series(test_df["lemma"].to_numpy()).map(cmap).to_numpy()

        out = test_df[["sent_id", "word_id", "repr_row", "lemma"]].copy()
        out["control_gold"] = yte
        summary = {"task": task, "T": len(test_df), "conditions": {}}
        for cond, keep in (("A", ["core", "seen"]), ("B", ["core", "reserve"])):
            mask = np.isin(part, keep)
            pipe, n_iter, converged = fit_probe(np.asarray(X[repr_row[mask]]), yctrl[mask])
            pred = pipe.predict(Xte)
            out[f"pred_{cond}"] = pred
            summary["conditions"][cond] = {"n_iter": n_iter, "converged": converged,
                                           "control": point_metrics(yte, pred)}
            del pipe

        out.to_parquet(pred_p, index=False)
        mk.write_text(json.dumps({"params": params, "summary": summary}, indent=2), encoding="utf-8")
        control_report[(code, task)] = summary
        A, B = summary["conditions"]["A"], summary["conditions"]["B"]
        print(f"{code:8s} {task:6s}: control acc A={A['control']['accuracy']:.3f} "
              f"B={B['control']['accuracy']:.3f}")
    del X
    gc.collect()

print("\nwrote cache/predictions/*_control.parquet")

en_ewt   upos  : control cache hit
en_ewt   number: control cache hit
it_isdt  upos  : control cache hit
it_isdt  number: control cache hit
pl_pdb   upos  : control cache hit
pl_pdb   number: control cache hit

wrote cache/predictions/*_control.parquet


In [23]:
# selectivity
ctrl_floor, sel_rows = {}, []
for (code, task) in main_report:
    gold = pd.read_parquet(PRED_DIR / f"{code}_{task}_control.parquet")["control_gold"]
    label = gold.value_counts().idxmax()
    ctrl_floor[(code, task)] = {"label": str(label),
                                **point_metrics(gold, np.full(len(gold), label))}
    for cond in ("A", "B"):
        t = main_report[(code, task)]["conditions"][cond]["probe"]
        c = control_report[(code, task)]["conditions"][cond]["control"]
        f = ctrl_floor[(code, task)]
        sel_rows.append({"language": code, "task": task, "cond": cond,
                         "task_acc": t["accuracy"], "ctrl_acc": c["accuracy"],
                         "sel_acc": t["accuracy"] - c["accuracy"],
                         "task_f1": t["macro_f1"], "ctrl_f1": c["macro_f1"],
                         "sel_f1": t["macro_f1"] - c["macro_f1"],
                         "ctrl_majority_label": f["label"],
                         "ctrl_majority_acc": f["accuracy"], "ctrl_majority_f1": f["macro_f1"]})
sel_tbl = pd.DataFrame(sel_rows)
print(sel_tbl.drop(columns=[c for c in sel_tbl.columns if c.startswith("ctrl_majority")])
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\ncontrol-label majority baseline on T (shared by A and B), vs the task floor:")
for (code, task), f in ctrl_floor.items():
    mt = main_report[(code, task)]["majority_test"]
    print(f"  {code:8s} {task:6s}: control {f['label']:5s} acc {f['accuracy']:.3f} / F1 {f['macro_f1']:.3f}"
          f"   |   task {mt['label']:5s} acc {mt['accuracy']:.3f} / F1 {mt['macro_f1']:.3f}")

(RESULTS_DIR / "selectivity_report.json").write_text(json.dumps(sel_rows, indent=2), encoding="utf-8")
print("\nwrote results/selectivity_report.json")

language   task cond  task_acc  ctrl_acc  sel_acc  task_f1  ctrl_f1  sel_f1
  en_ewt   upos    A     0.959     0.630    0.329    0.919    0.545   0.374
  en_ewt   upos    B     0.866     0.103    0.763    0.677    0.058   0.620
  en_ewt number    A     0.991     0.796    0.195    0.988    0.622   0.366
  en_ewt number    B     0.986     0.649    0.337    0.981    0.467   0.515
 it_isdt   upos    A     0.977     0.738    0.239    0.860    0.586   0.274
 it_isdt   upos    B     0.839     0.078    0.760    0.683    0.036   0.647
 it_isdt number    A     0.996     0.752    0.244    0.995    0.668   0.327
 it_isdt number    B     0.991     0.608    0.383    0.990    0.478   0.512
  pl_pdb   upos    A     0.981     0.594    0.387    0.905    0.510   0.395
  pl_pdb   upos    B     0.847     0.137    0.710    0.622    0.050   0.572
  pl_pdb number    A     0.989     0.756    0.233    0.986    0.579   0.408
  pl_pdb number    B     0.985     0.683    0.302    0.982    0.479   0.503

control-lab

In [24]:
# gap and its paired bootstrap CI
def boot_seed(code, task):
    return int.from_bytes(hashlib.sha256(f"{SEED}:bootstrap:{code}:{task}".encode()).digest()[:8], "big")


def macro_f1_codes(gc_, pc_, K):
    """Fast macro-F1 over integer-coded labels; matches sklearn's macro average."""
    correct = gc_ == pc_
    tp = np.bincount(gc_[correct], minlength=K).astype(np.float64)
    actual = np.bincount(gc_, minlength=K).astype(np.float64)
    predicted = np.bincount(pc_, minlength=K).astype(np.float64)
    prec = np.divide(tp, predicted, out=np.zeros(K), where=predicted > 0)
    rec = np.divide(tp, actual, out=np.zeros(K), where=actual > 0)
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros(K), where=denom > 0)
    present = (actual > 0) | (predicted > 0)
    return float(f1[present].mean()) if present.any() else 0.0


B_RESAMPLES = CONFIG["evaluation"]["bootstrap_resamples"]
LO, HI = CONFIG["evaluation"]["ci_percentiles"]
gap_report = {}
for (code, task) in main_report:
    dfp = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet")
    gold, predA, predB = dfp["gold_label"].to_numpy(), dfp["pred_A"].to_numpy(), dfp["pred_B"].to_numpy()
    cats = pd.CategoricalDtype(sorted(set(gold) | set(predA) | set(predB)))
    gc_ = pd.Categorical(gold, dtype=cats).codes.astype(np.int64)
    aca = pd.Categorical(predA, dtype=cats).codes.astype(np.int64)
    acb = pd.Categorical(predB, dtype=cats).codes.astype(np.int64)
    K = len(cats.categories)
    corrA, corrB = (gc_ == aca), (gc_ == acb)

    # fast macro-F1 must match sklearn
    assert abs(macro_f1_codes(gc_, aca, K) - f1_score(gold, predA, average="macro", zero_division=0)) < 1e-9
    d_acc = float(corrA.mean() - corrB.mean())
    d_f1 = macro_f1_codes(gc_, aca, K) - macro_f1_codes(gc_, acb, K)

    # both conditions on same resampled indices
    rng = np.random.default_rng(boot_seed(code, task))
    T = len(gold)
    dacc_vec = corrA.astype(np.int16) - corrB.astype(np.int16)
    b_acc, b_f1 = np.empty(B_RESAMPLES), np.empty(B_RESAMPLES)
    for i in range(B_RESAMPLES):
        idx = rng.integers(0, T, T)
        b_acc[i] = dacc_vec[idx].mean()
        b_f1[i] = macro_f1_codes(gc_[idx], aca[idx], K) - macro_f1_codes(gc_[idx], acb[idx], K)

    gap_report[(code, task)] = {
        "delta_acc": d_acc, "acc_ci": [float(np.percentile(b_acc, LO)), float(np.percentile(b_acc, HI))],
        "delta_f1": d_f1, "f1_ci": [float(np.percentile(b_f1, LO)), float(np.percentile(b_f1, HI))]}
    r = gap_report[(code, task)]
    print(f"{code:8s} {task:6s}: dAcc={d_acc:+.3f} [{r['acc_ci'][0]:+.3f}, {r['acc_ci'][1]:+.3f}]"
          f"   dMacroF1={d_f1:+.3f} [{r['f1_ci'][0]:+.3f}, {r['f1_ci'][1]:+.3f}]")

en_ewt   upos  : dAcc=+0.093 [+0.090, +0.096]   dMacroF1=+0.242 [+0.233, +0.250]
en_ewt   number: dAcc=+0.005 [+0.002, +0.007]   dMacroF1=+0.007 [+0.003, +0.010]
it_isdt  upos  : dAcc=+0.138 [+0.135, +0.141]   dMacroF1=+0.177 [+0.161, +0.199]
it_isdt  number: dAcc=+0.005 [+0.003, +0.006]   dMacroF1=+0.005 [+0.003, +0.007]
pl_pdb   upos  : dAcc=+0.135 [+0.132, +0.137]   dMacroF1=+0.283 [+0.231, +0.294]
pl_pdb   number: dAcc=+0.004 [+0.002, +0.005]   dMacroF1=+0.005 [+0.003, +0.006]


In [25]:
# McNemar on A-vs-B correctness, then Holm
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

mcnemar_report = {}
for (code, task) in main_report:
    dfp = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet")
    gold = dfp["gold_label"].to_numpy()
    cA, cB = (dfp["pred_A"].to_numpy() == gold), (dfp["pred_B"].to_numpy() == gold)
    a = int((cA & cB).sum()); b = int((cA & ~cB).sum())
    c = int((~cA & cB).sum()); dd = int((~cA & ~cB).sum())
    exact = (b + c) < 25   # exact binomial when discordants are few
    res = mcnemar([[a, b], [c, dd]], exact=exact, correction=not exact)
    mcnemar_report[(code, task)] = {"b_A_correct_B_wrong": b, "c_A_wrong_B_correct": c,
                                    "test": "exact" if exact else "chi2_continuity",
                                    "statistic": float(res.statistic), "p_raw": float(res.pvalue)}

keys = list(mcnemar_report)
_, p_holm, _, _ = multipletests([mcnemar_report[k]["p_raw"] for k in keys], method="holm")
for k, ph in zip(keys, p_holm):
    mcnemar_report[k]["p_holm"] = float(ph)

for (code, task) in keys:
    m = mcnemar_report[(code, task)]
    print(f"{code:8s} {task:6s}: b={m['b_A_correct_B_wrong']:>6,} c={m['c_A_wrong_B_correct']:>6,}  "
          f"{m['test']:16s} stat={m['statistic']:>12.1f}  p_raw={m['p_raw']:.2e}  p_holm={m['p_holm']:.2e}")

en_ewt   upos  : b= 5,019 c=   395  chi2_continuity  stat=      3947.6  p_raw=0.00e+00  p_holm=0.00e+00
en_ewt   number: b=    70 c=    30  chi2_continuity  stat=        15.2  p_raw=9.62e-05  p_holm=9.62e-05
it_isdt  upos  : b= 7,392 c=   254  chi2_continuity  stat=      6661.9  p_raw=0.00e+00  p_holm=0.00e+00
it_isdt  number: b=    70 c=    19  chi2_continuity  stat=        28.1  p_raw=1.16e-07  p_holm=3.47e-07
pl_pdb   upos  : b=10,168 c=   352  chi2_continuity  stat=      9157.2  p_raw=0.00e+00  p_holm=0.00e+00
pl_pdb   number: b=   113 c=    49  chi2_continuity  stat=        24.5  p_raw=7.43e-07  p_holm=1.49e-06


In [26]:
# results tables
main_rows = []
for (code, task) in main_report:
    s = split_summaries[(code, task)]
    mt = main_report[(code, task)]["majority_test"]
    for cond in ("A", "B"):
        mp = main_report[(code, task)]["conditions"][cond]
        cp = control_report[(code, task)]["conditions"][cond]["control"]
        main_rows.append({
            "language": code, "task": task, "condition": cond,
            "n_train": mp["n_train"], "n_test": main_report[(code, task)]["T"],
            "accuracy": mp["probe"]["accuracy"], "macro_f1": mp["probe"]["macro_f1"],
            "majority_test_label": mt["label"],
            "majority_test_acc": mt["accuracy"], "majority_test_f1": mt["macro_f1"],
            "control_acc": cp["accuracy"], "control_f1": cp["macro_f1"],
            "control_majority_label": ctrl_floor[(code, task)]["label"],
            "control_majority_acc": ctrl_floor[(code, task)]["accuracy"],
            "control_majority_f1": ctrl_floor[(code, task)]["macro_f1"],
            "selectivity_acc": mp["probe"]["accuracy"] - cp["accuracy"],
            "selectivity_f1": mp["probe"]["macro_f1"] - cp["macro_f1"],
            "train_label_dist": json.dumps(s[f"train_{cond}_by_class"]),
            "test_label_dist": json.dumps(s["T_by_class"]),
        })
pd.DataFrame(main_rows).to_csv(RESULTS_DIR / "main_results.csv", index=False)

gap_out = [{"language": c, "task": t, "delta_acc": v["delta_acc"],
            "acc_ci_lo": v["acc_ci"][0], "acc_ci_hi": v["acc_ci"][1],
            "delta_macro_f1": v["delta_f1"], "f1_ci_lo": v["f1_ci"][0], "f1_ci_hi": v["f1_ci"][1],
            "bootstrap_resamples": B_RESAMPLES} for (c, t), v in gap_report.items()]
pd.DataFrame(gap_out).to_csv(RESULTS_DIR / "gap_table.csv", index=False)

pd.DataFrame([{"language": c, "task": t, **mcnemar_report[(c, t)]}
              for (c, t) in mcnemar_report]).to_csv(RESULTS_DIR / "mcnemar_table.csv", index=False)

print("wrote results/main_results.csv, results/gap_table.csv, results/mcnemar_table.csv\n")
print(pd.DataFrame(gap_out)[["language", "task", "delta_acc", "acc_ci_lo", "acc_ci_hi",
                              "delta_macro_f1", "f1_ci_lo", "f1_ci_hi"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

wrote results/main_results.csv, results/gap_table.csv, results/mcnemar_table.csv

language   task  delta_acc  acc_ci_lo  acc_ci_hi  delta_macro_f1  f1_ci_lo  f1_ci_hi
  en_ewt   upos      0.093      0.090      0.096           0.242     0.233     0.250
  en_ewt number      0.005      0.002      0.007           0.007     0.003     0.010
 it_isdt   upos      0.138      0.135      0.141           0.177     0.161     0.199
 it_isdt number      0.005      0.003      0.006           0.005     0.003     0.007
  pl_pdb   upos      0.135      0.132      0.137           0.283     0.231     0.294
  pl_pdb number      0.004      0.002      0.005           0.005     0.003     0.006


## 8. Fragmentation Analysis and Exhibits

Fits the error-versus-fragmentation logistic regression on condition-B test occurrences, per cell and pooled per task, recording VIFs. Then the per-class F1 table and its support check. Writes `results/error_analysis_*.csv` and `per_class_f1*.csv`.

In [27]:
# fragmentation vs error, per cell
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import PerfectSeparationError

# form frequency over pooled tokens, add-one smoothed
freq_by_code = {}
for code in CONFIG["treebanks"].values():
    m = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet",
                        columns=["sent_id", "word_id", "form", "n_subwords", "char_length"])
    freq_by_code[code] = (m, m["form"].value_counts())


def zscore(x):
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=0)
    return (x - x.mean()) / sd if sd > 0 else np.zeros_like(x)


def build_frame(code, task):
    preds = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet")
    meta, freq = freq_by_code[code]
    df = preds.merge(meta[["sent_id", "word_id", "form", "n_subwords", "char_length"]],
                     on=["sent_id", "word_id"], how="left")
    df["error"] = (df["pred_B"].to_numpy() != df["gold_label"].to_numpy()).astype(int)
    df["log_freq"] = np.log(df["form"].map(freq).fillna(0).to_numpy() + 1.0)
    df["z_n_subwords"] = zscore(df["n_subwords"])
    df["z_char_length"] = zscore(df["char_length"])
    df["z_log_freq"] = zscore(df["log_freq"])
    df["label"] = df["gold_label"]
    return df


err_frames = {(code, task): build_frame(code, task)
              for code in CONFIG["treebanks"].values() for task in ("upos", "number")}

coef_rows, vif_rows = [], []
FORMULA = "error ~ z_n_subwords + z_char_length + z_log_freq + C(label)"
for (code, task), df in err_frames.items():
    n_err = int(df["error"].sum())
    try:
        res = smf.logit(FORMULA, data=df).fit(method="bfgs", maxiter=300, disp=0)
        label_in = True
    except (PerfectSeparationError, np.linalg.LinAlgError, ValueError):
        res = smf.logit("error ~ z_n_subwords + z_char_length + z_log_freq", data=df).fit(
            method="bfgs", maxiter=300, disp=0)
        label_in = False
    converged = bool(res.mle_retvals["converged"])   # a stopped optimizer is not an estimate
    # all-or-none errors quasi-separate a class: its coefficient is not identified
    by_label = df.groupby("label")["error"].agg(n="size", errors="sum")
    ci = res.conf_int()
    for term in res.params.index:
        lab = term.removeprefix("C(label)[T.").removesuffix("]") if term.startswith("C(label)[T.") else None
        n_lab, e_lab = ((int(by_label.loc[lab, "n"]), int(by_label.loc[lab, "errors"]))
                        if lab is not None else (None, None))
        coef_rows.append({"language": code, "task": task, "term": term,
                          "log_odds": float(res.params[term]),
                          "ci_lo": float(ci.loc[term, 0]), "ci_hi": float(ci.loc[term, 1]),
                          "p_value": float(res.pvalues[term]),
                          "headline": term == "z_n_subwords", "C_label_included": label_in,
                          "converged": converged, "class_n": n_lab, "class_errors": e_lab,
                          "all_or_none": None if lab is None else e_lab in (0, n_lab)})
    Xc = np.column_stack([np.ones(len(df)), df["z_n_subwords"], df["z_char_length"], df["z_log_freq"]])
    for j, name in enumerate(["z_n_subwords", "z_char_length", "z_log_freq"], start=1):
        vif_rows.append({"language": code, "task": task, "predictor": name,
                         "vif": float(variance_inflation_factor(Xc, j))})
    print(f"{code:8s} {task:6s}: n_err={n_err:>6,}  n_subwords log-odds={res.params['z_n_subwords']:+.3f} "
          f"[{ci.loc['z_n_subwords', 0]:+.3f}, {ci.loc['z_n_subwords', 1]:+.3f}] "
          f"p={res.pvalues['z_n_subwords']:.2e}  (C(label) {'in' if label_in else 'DROPPED: separation'})")

stopped = sorted({(r["language"], r["task"]) for r in coef_rows if not r["converged"]})
print("\nBFGS (maxiter 300):", "all 6 fits converged" if not stopped else f"did not converge: {stopped}")
flagged = [r for r in coef_rows if r["all_or_none"]]
print("class coefficients not identified (all-or-none errors):",
      ", ".join(f"{r['language']}/{r['task']} "
                f"{r['term'].removeprefix('C(label)[T.').removesuffix(']')} "
                f"({r['class_errors']}/{r['class_n']})" for r in flagged) or "none")

pd.DataFrame(coef_rows).to_csv(RESULTS_DIR / "error_analysis_coefficients.csv", index=False)
pd.DataFrame(vif_rows).to_csv(RESULTS_DIR / "error_analysis_vif.csv", index=False)
print("\nVIF (continuous predictors):")
print(pd.DataFrame(vif_rows).pivot(index=["language", "task"], columns="predictor", values="vif")
      .round(2).to_string())
print("\nwrote results/error_analysis_coefficients.csv, results/error_analysis_vif.csv")

en_ewt   upos  : n_err= 6,647  n_subwords log-odds=+0.114 [+0.080, +0.148] p=7.89e-11  (C(label) in)
en_ewt   number: n_err=   115  n_subwords log-odds=-0.126 [-0.405, +0.152] p=3.75e-01  (C(label) in)
it_isdt  upos  : n_err= 8,330  n_subwords log-odds=+0.012 [-0.028, +0.052] p=5.61e-01  (C(label) in)
it_isdt  number: n_err=    96  n_subwords log-odds=+1.330 [+1.073, +1.588] p=4.73e-24  (C(label) in)
pl_pdb   upos  : n_err=11,180  n_subwords log-odds=-0.408 [-0.464, -0.352] p=2.18e-46  (C(label) in)
pl_pdb   number: n_err=   252  n_subwords log-odds=-0.112 [-0.284, +0.059] p=2.00e-01  (C(label) in)

BFGS (maxiter 300): all 6 fits converged
class coefficients not identified (all-or-none errors): it_isdt/upos INTJ (0/5), it_isdt/upos SYM (2/2), pl_pdb/upos SYM (2/2)

VIF (continuous predictors):
predictor        z_char_length  z_log_freq  z_n_subwords
language task                                           
en_ewt   number           1.18        1.40          1.43
         upos           

In [28]:
# supplement: pooled across languages, per task
pooled_rows = []
for task in ("upos", "number"):
    pool = pd.concat([err_frames[(code, task)].assign(language=code)
                      for code in CONFIG["treebanks"].values()], ignore_index=True)
    for col in ("n_subwords", "char_length", "log_freq"):
        pool[f"z_{col}"] = zscore(pool[col])   # re-z-scored over pooled data
    try:
        res = smf.logit("error ~ z_n_subwords + z_char_length + z_log_freq + C(label) + C(language)",
                        data=pool).fit(method="bfgs", maxiter=300, disp=0)
        label_in = True
    except (PerfectSeparationError, np.linalg.LinAlgError, ValueError):
        res = smf.logit("error ~ z_n_subwords + z_char_length + z_log_freq + C(language)",
                        data=pool).fit(method="bfgs", maxiter=300, disp=0)
        label_in = False
    converged = bool(res.mle_retvals["converged"])
    by_label = pool.groupby("label")["error"].agg(n="size", errors="sum")
    ci = res.conf_int()
    for term in res.params.index:
        lab = term.removeprefix("C(label)[T.").removesuffix("]") if term.startswith("C(label)[T.") else None
        n_lab, e_lab = ((int(by_label.loc[lab, "n"]), int(by_label.loc[lab, "errors"]))
                        if lab is not None else (None, None))
        pooled_rows.append({"task": task, "term": term, "log_odds": float(res.params[term]),
                            "ci_lo": float(ci.loc[term, 0]), "ci_hi": float(ci.loc[term, 1]),
                            "p_value": float(res.pvalues[term]),
                            "headline": term == "z_n_subwords", "C_label_included": label_in,
                            "converged": converged, "class_n": n_lab, "class_errors": e_lab,
                            "all_or_none": None if lab is None else e_lab in (0, n_lab)})
    print(f"pooled {task:6s} (N={len(pool):>7,}): n_subwords log-odds={res.params['z_n_subwords']:+.3f} "
          f"[{ci.loc['z_n_subwords', 0]:+.3f}, {ci.loc['z_n_subwords', 1]:+.3f}] "
          f"p={res.pvalues['z_n_subwords']:.2e}  (C(label) {'in' if label_in else 'dropped'})")
flagged = [r for r in pooled_rows if r["all_or_none"]]
print("class coefficients not identified (all-or-none errors):",
      ", ".join(f"{r['task']} {r['term'].removeprefix('C(label)[T.').removesuffix(']')} "
                f"({r['class_errors']}/{r['class_n']})" for r in flagged) or "none")
stopped = sorted({r["task"] for r in pooled_rows if not r["converged"]})
print("\nBFGS (maxiter 300):", "both pooled fits converged" if not stopped else f"did not converge: {stopped}")

pd.DataFrame(pooled_rows).to_csv(RESULTS_DIR / "error_analysis_pooled.csv", index=False)
print("\nwrote results/error_analysis_pooled.csv")

pooled upos   (N=174,180): n_subwords log-odds=-0.198 [-0.225, -0.171] p=4.57e-46  (C(label) in)
pooled number (N= 36,654): n_subwords log-odds=+0.179 [+0.060, +0.298] p=3.21e-03  (C(label) in)
class coefficients not identified (all-or-none errors): none

BFGS (maxiter 300): both pooled fits converged

wrote results/error_analysis_pooled.csv


In [29]:
#  macro-F1 gap by gold class
from scipy.stats import spearmanr

# NUM sits with the open classes
OPEN_UPOS = {"ADJ", "ADV", "INTJ", "NOUN", "NUM", "PROPN", "VERB"}
CLOSED_UPOS = {"ADP", "AUX", "CCONJ", "DET", "PART", "PRON", "SCONJ"}


def upos_class_type(label, num_closed=False):
    if label == "NUM":
        return "closed" if num_closed else "open"
    return "closed" if label in CLOSED_UPOS else "open" if label in OPEN_UPOS else "other"


per_class_rows, support_rows = [], []
for (code, task) in main_report:
    dfp = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet")
    gold, pA, pB = dfp["gold_label"], dfp["pred_A"], dfp["pred_B"]
    classes = sorted(set(gold) | set(pA) | set(pB))   # realized classes
    f1A = f1_score(gold, pA, average=None, labels=classes, zero_division=0)
    f1B = f1_score(gold, pB, average=None, labels=classes, zero_division=0)
    delta = f1A - f1B
    support = gold.value_counts().reindex(classes).fillna(0).astype(int).to_numpy()

    # per-class differences must average to reported gap
    assert abs(float(delta.mean()) - gap_report[(code, task)]["delta_f1"]) < 1e-9

    for lab, a, b, d, n in zip(classes, f1A, f1B, delta, support):
        per_class_rows.append({"language": code, "task": task, "label": lab,
                               "support_T": int(n), "f1_A": float(a), "f1_B": float(b),
                               "delta_f1": float(d),
                               "class_type": upos_class_type(lab) if task == "upos" else "-"})

    # rank correlation and median split: either alone can be carried by one class
    rho, p = spearmanr(support, delta) if len(classes) >= 4 else (np.nan, np.nan)
    med = float(np.median(support))
    support_rows.append({"language": code, "task": task, "n_classes": len(classes),
                         "spearman_support_delta": float(rho), "spearman_p": float(p),
                         "mean_delta_low_support": float(delta[support <= med].mean()),
                         "mean_delta_high_support": float(delta[support > med].mean()),
                         "macro_f1_gap": float(delta.mean())})

per_class = pd.DataFrame(per_class_rows)
support_check = pd.DataFrame(support_rows)
per_class.to_csv(RESULTS_DIR / "per_class_f1.csv", index=False)
support_check.to_csv(RESULTS_DIR / "per_class_f1_support_check.csv", index=False)

print("is the macro-F1 gap carried by the rare classes? (UPOS; over classes within a cell)")
print(support_check[support_check["task"] == "upos"]
      .drop(columns=["task"]).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nthree largest per-class drops per UPOS cell:")
for code in CONFIG["treebanks"].values():
    top = per_class[(per_class["language"] == code)
                    & (per_class["task"] == "upos")].nlargest(3, "delta_f1")
    print(f"  {code:8s} " + "   ".join(f"{r.label} (n={r.support_T:,}) {r.delta_f1:+.3f}"
                                       for r in top.itertuples()))
print("\nwrote results/per_class_f1.csv, results/per_class_f1_support_check.csv")


is the macro-F1 gap carried by the rare classes? (UPOS; over classes within a cell)
language  n_classes  spearman_support_delta  spearman_p  mean_delta_low_support  mean_delta_high_support  macro_f1_gap
  en_ewt         17                  -0.841       0.000                   0.389                    0.076         0.242
 it_isdt         16                  -0.085       0.753                   0.249                    0.105         0.177
  pl_pdb         17                  -0.471       0.057                   0.321                    0.239         0.283

three largest per-class drops per UPOS cell:
  en_ewt   PART (n=597) +0.988   DET (n=449) +0.584   INTJ (n=303) +0.418
  it_isdt  SCONJ (n=94) +0.547   X (n=21) +0.547   PRON (n=1,358) +0.507
  pl_pdb   SYM (n=2) +1.000   AUX (n=2,513) +0.637   PART (n=1,390) +0.436

wrote results/per_class_f1.csv, results/per_class_f1_support_check.csv


### Validity Addenda

Three exhibits for Threats to Validity: share of the gap carried by the five largest held-out lemmata with its leave-one-out range, share of condition-B test tokens whose form also occurs in `train_B`, and the gap by open versus closed class. Writes `results/h_sensitivity_*.csv`, `form_overlap.csv`, and `open_closed_gap.csv`.

In [30]:
# agap on T minus one held-out lemma at a time, against CI width
TOP_K = 10
loo_rows, hsum_rows = [], []

for (code, task) in main_report:
    dfp = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet")
    gold, predA, predB = dfp["gold_label"].to_numpy(), dfp["pred_A"].to_numpy(), dfp["pred_B"].to_numpy()
    lemma = dfp["lemma"].to_numpy()
    cats = pd.CategoricalDtype(sorted(set(gold) | set(predA) | set(predB)))
    gc_ = pd.Categorical(gold, dtype=cats).codes.astype(np.int64)
    aca = pd.Categorical(predA, dtype=cats).codes.astype(np.int64)
    acb = pd.Categorical(predB, dtype=cats).codes.astype(np.int64)
    K = len(cats.categories)
    corrA, corrB = (gc_ == aca), (gc_ == acb)
    T = len(gold)

    def deltas(mask):
        return (float(corrA[mask].mean() - corrB[mask].mean()),
                macro_f1_codes(gc_[mask], aca[mask], K) - macro_f1_codes(gc_[mask], acb[mask], K))

    def floor(mask):
        vc = pd.Series(gold[mask]).value_counts()
        return str(vc.index[0]), float(vc.iloc[0] / mask.sum())

    full = np.ones(T, dtype=bool)
    d_acc_full, d_f1_full = deltas(full)
    # with nothing deleted, this must reduce to the reported gap
    assert abs(d_acc_full - gap_report[(code, task)]["delta_acc"]) < 1e-9
    assert abs(d_f1_full - gap_report[(code, task)]["delta_f1"]) < 1e-9

    counts = pd.Series(lemma).value_counts()
    top = counts.head(TOP_K)
    for rank, (lem, n) in enumerate(top.items(), start=1):
        keep = lemma != lem
        d_acc, d_f1 = deltas(keep)
        lab, acc = floor(keep)
        loo_rows.append({"language": code, "task": task, "rank": rank, "lemma": lem,
                         "n_test_occ": int(n), "share_T": float(n / T),
                         "delta_acc_without": d_acc, "delta_f1_without": d_f1,
                         "shift_acc": d_acc - d_acc_full, "shift_f1": d_f1 - d_f1_full,
                         "floor_label_without": lab, "floor_acc_without": acc})

    in5 = np.isin(lemma, list(top.head(5).index))
    d5_acc, d5_f1 = deltas(in5)
    dr_acc, dr_f1 = deltas(~in5)
    cell = [r for r in loo_rows if r["language"] == code and r["task"] == task]
    acc_w = float(np.diff(gap_report[(code, task)]["acc_ci"])[0])
    f1_w = float(np.diff(gap_report[(code, task)]["f1_ci"])[0])
    rng_acc = max(r["delta_acc_without"] for r in cell) - min(r["delta_acc_without"] for r in cell)
    rng_f1 = max(r["delta_f1_without"] for r in cell) - min(r["delta_f1_without"] for r in cell)
    lab_full, acc_full = floor(full)
    hsum_rows.append({"language": code, "task": task, "n_H_lemmas": int(counts.size), "T": int(T),
                      "top5_share_T": float(top.head(5).sum() / T),
                      "delta_acc_top5": d5_acc, "delta_acc_rest": dr_acc,
                      "delta_f1_top5": d5_f1, "delta_f1_rest": dr_f1,
                      "delta_acc_full": d_acc_full, "delta_f1_full": d_f1_full,
                      "loo_range_acc": rng_acc, "loo_range_f1": rng_f1,
                      "max_abs_shift_acc": max(abs(r["shift_acc"]) for r in cell),
                      "max_abs_shift_f1": max(abs(r["shift_f1"]) for r in cell),
                      "boot_ci_width_acc": acc_w, "boot_ci_width_f1": f1_w,
                      "ratio_acc": rng_acc / acc_w, "ratio_f1": rng_f1 / f1_w,
                      "floor_label": lab_full, "floor_acc": acc_full,
                      "n_floor_flips_top10": sum(1 for r in cell
                                                 if r["floor_label_without"] != lab_full)})

h_loo, h_sum = pd.DataFrame(loo_rows), pd.DataFrame(hsum_rows)
h_loo.to_csv(RESULTS_DIR / "h_sensitivity_leave_one_out.csv", index=False)
h_sum.to_csv(RESULTS_DIR / "h_sensitivity_summary.csv", index=False)

print("how much of the gap sits on the five largest held-out lemmata, and how far one")
print("deletion moves it, against the bootstrap interval that holds H fixed:")
print(h_sum[["language", "task", "top5_share_T", "delta_acc_top5", "delta_acc_rest",
             "loo_range_acc", "boot_ci_width_acc", "ratio_acc", "ratio_f1"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nfive largest held-out lemmata, UPOS cells:")
for code in CONFIG["treebanks"].values():
    sub = h_loo[(h_loo["language"] == code) & (h_loo["task"] == "upos")].head(5)
    print(f"  {code:8s} " + "  ".join(f"{r.lemma!r} {r.share_T:.1%}" for r in sub.itertuples()))
print("\nlargest single-lemma shift per UPOS cell:")
for code in CONFIG["treebanks"].values():
    sub = h_loo[(h_loo["language"] == code) & (h_loo["task"] == "upos")]
    wa = sub.loc[sub["shift_acc"].abs().idxmax()]
    wf = sub.loc[sub["shift_f1"].abs().idxmax()]
    print(f"  {code:8s} acc {wa.shift_acc:+.4f} without {wa.lemma!r}"
          f"   macro-F1 {wf.shift_f1:+.4f} without {wf.lemma!r}"
          f" ({gap_report[(code, 'upos')]['delta_f1']:.3f} -> {wf.delta_f1_without:.3f})")
print("\nmajority floor is itself a property of the draw:")
for r in h_sum.itertuples():
    if r.n_floor_flips_top10:
        flips = h_loo[(h_loo["language"] == r.language) & (h_loo["task"] == r.task)
                      & (h_loo["floor_label_without"] != r.floor_label)]
        print(f"  {r.language:8s} {r.task:6s} {r.floor_label} ({r.floor_acc:.3f}) -> "
              + ", ".join(f"{x.floor_label_without} ({x.floor_acc_without:.3f}) without {x.lemma!r}"
                          for x in flips.itertuples()))
print("\nwrote results/h_sensitivity_leave_one_out.csv, results/h_sensitivity_summary.csv")

how much of the gap sits on the five largest held-out lemmata, and how far one
deletion moves it, against the bootstrap interval that holds H fixed:
language   task  top5_share_T  delta_acc_top5  delta_acc_rest  loo_range_acc  boot_ci_width_acc  ratio_acc  ratio_f1
  en_ewt   upos        0.1848          0.0758          0.0971         0.0136             0.0056     2.4478    3.7733
  en_ewt number        0.0449          0.0314          0.0034         0.0009             0.0046     0.2051    0.2021
 it_isdt   upos        0.4821          0.2433          0.0401         0.0298             0.0063     4.7594    1.1707
 it_isdt number        0.0784          0.0476          0.0010         0.0044             0.0034     1.3147    1.3135
  pl_pdb   upos        0.2301          0.2892          0.0886         0.0221             0.0052     4.2759    0.4473
  pl_pdb number        0.0868          0.0060          0.0035         0.0005             0.0029     0.1715    0.1927

five largest held-out lemmata, 

In [31]:
# disjoint in lemma types, not in surface forms; threshold is on train_B side
THRESHOLDS = (1, 20)
form_rows, meta_forms = [], {}

for (code, task) in main_report:
    if code not in meta_forms:
        meta_forms[code] = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet").set_index("row")["form"]
    dfs = pd.read_parquet(SPLIT_DIR / f"{code}_{task}.parquet")
    dfs = dfs.assign(form=meta_forms[code].reindex(dfs["repr_row"].to_numpy()).to_numpy())
    assert dfs["form"].notna().all()

    train_B = dfs[dfs["partition"].isin(("core", "reserve"))]
    test = dfs[dfs["partition"] == "test"]
    # no test lemma occurs in train_B
    assert not (set(test["lemma"]) & set(train_B["lemma"]))

    fc = train_B["form"].value_counts()
    seen = fc.reindex(test["form"].to_numpy()).fillna(0).to_numpy()
    row = {"language": code, "task": task, "n_test": len(test), "n_train_B": len(train_B),
           "n_distinct_test_forms": int(test["form"].nunique())}
    for th in THRESHOLDS:
        row[f"share_form_seen_ge{th}"] = float((seen >= th).mean())
        row[f"n_form_seen_ge{th}"] = int((seen >= th).sum())

    # the single form covering the most test tokens
    hits = test.assign(c=seen).query("c >= 20")["form"].value_counts()
    if len(hits):
        row["top_form_ge20"] = str(hits.index[0])
        row["top_form_trainB_count"] = int(fc[hits.index[0]])
        row["top_form_share_of_test"] = float(hits.iloc[0] / len(test))
        row["top_form_share_of_ge20"] = float(hits.iloc[0] / hits.sum())
        row["shared_forms_ge20"] = "; ".join(
            f"{f} ({int(fc[f])} in train_B, {int(n)} test tokens)" for f, n in hits.head(5).items())
    else:
        row.update({"top_form_ge20": "", "top_form_trainB_count": 0,
                    "top_form_share_of_test": 0.0, "top_form_share_of_ge20": 0.0,
                    "shared_forms_ge20": ""})
    form_rows.append(row)

form_overlap = pd.DataFrame(form_rows)
form_overlap.to_csv(RESULTS_DIR / "form_overlap.csv", index=False)

print("share of condition-B test tokens whose surface form also occurs in train_B:")
print(form_overlap[["language", "task", "n_test", "share_form_seen_ge1", "n_form_seen_ge1",
                    "share_form_seen_ge20", "n_form_seen_ge20"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nthe >= 20 column is thin: one form carries most of it, on little training support")
print(form_overlap[["language", "task", "top_form_ge20", "top_form_trainB_count",
                    "top_form_share_of_test", "top_form_share_of_ge20"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nforms shared at >= 20 train_B occurrences, UPOS cells:")
for r in form_overlap[form_overlap["task"] == "upos"].itertuples():
    print(f"  {r.language:8s} {r.shared_forms_ge20}")
print("\nwrote results/form_overlap.csv")

share of condition-B test tokens whose surface form also occurs in train_B:
language   task  n_test  share_form_seen_ge1  n_form_seen_ge1  share_form_seen_ge20  n_form_seen_ge20
  en_ewt   upos   49630               0.2302            11425                0.0025               124
  en_ewt number    8510               0.0038               32                0.0000                 0
 it_isdt   upos   51709               0.1834             9485                0.0840              4345
 it_isdt number   10989               0.0338              371                0.0015                17
  pl_pdb   upos   72841               0.1054             7676                0.0684              4984
  pl_pdb number   17155               0.0242              416                0.0007                12

the >= 20 column is thin: one form carries most of it, on little training support
language   task top_form_ge20  top_form_trainB_count  top_form_share_of_test  top_form_share_of_ge20
  en_ewt   upos        bet

In [32]:
# share of the gap rather than a rank test, which misses in English
from scipy.stats import mannwhitneyu

oc_rows = []
for variant, num_closed in (("NUM_open", False), ("NUM_closed", True)):
    for code in CONFIG["treebanks"].values():
        sub = per_class[(per_class["language"] == code) & (per_class["task"] == "upos")]
        ct = np.array([upos_class_type(lab, num_closed) for lab in sub["label"]])
        cl = sub["delta_f1"].to_numpy()[ct == "closed"]
        op = sub["delta_f1"].to_numpy()[ct == "open"]
        u, p = mannwhitneyu(cl, op, alternative="greater")
        oc_rows.append({"variant": variant, "language": code, "n_classes": len(sub),
                        "n_closed": len(cl), "n_open": len(op),
                        "closed_mean_delta_f1": float(cl.mean()),
                        "open_mean_delta_f1": float(op.mean()),
                        "closed_share_of_gap": float(cl.sum() / sub["delta_f1"].sum()),
                        "closed_share_of_classes": len(cl) / len(sub),
                        "macro_f1_gap": float(sub["delta_f1"].mean()),
                        "mannwhitney_u": float(u), "mannwhitney_p": float(p)})

open_closed = pd.DataFrame(oc_rows)
open_closed.to_csv(RESULTS_DIR / "open_closed_gap.csv", index=False)

# quoted share must not depend on where NUM goes
assert (open_closed.groupby("language")["closed_share_of_gap"].agg(lambda s: s.max() - s.min())
        < 0.03).all()

print("closed classes supply what share of the UPOS macro-F1 gap, from what share of the classes:")
print(open_closed.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nwrote results/open_closed_gap.csv, results/per_class_f1.csv (+ class_type column)")

closed classes supply what share of the UPOS macro-F1 gap, from what share of the classes:
   variant language  n_classes  n_closed  n_open  closed_mean_delta_f1  open_mean_delta_f1  closed_share_of_gap  closed_share_of_classes  macro_f1_gap  mannwhitney_u  mannwhitney_p
  NUM_open   en_ewt         17         7       7                 0.338               0.140                0.575                    0.412         0.242         34.000          0.130
  NUM_open  it_isdt         16         6       7                 0.280               0.063                0.591                    0.375         0.177         39.000          0.004
  NUM_open   pl_pdb         17         7       7                 0.371               0.138                0.540                    0.412         0.283         43.000          0.009
NUM_closed   en_ewt         17         8       6                 0.309               0.145                0.601                    0.471         0.242         32.000          0.172
NUM_

## 9. Figures and Artifact Index

Draws the six Appendix E figures and indexes the artifacts written so far. Writes `results/figures/*.png` and `results/artifacts_index.json`.

In [33]:
# Appendix E figures
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DPI = 300
LANGS = list(CONFIG["treebanks"].values())
TASK_TITLE = {"upos": "UPOS", "number": "Number"}
cells = [(c, t) for c in LANGS for t in ("upos", "number")]
labels = [f"{c}\n{t}" for (c, t) in cells]
xpos = np.arange(len(cells))


def grouped(metric_key, fname):
    A = [main_report[(c, t)]["conditions"]["A"]["probe"][metric_key] for (c, t) in cells]
    Bv = [main_report[(c, t)]["conditions"]["B"]["probe"][metric_key] for (c, t) in cells]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(xpos - 0.2, A, 0.4, label="A (lexical overlap)", color="#4C72B0")
    ax.bar(xpos + 0.2, Bv, 0.4, label="B (lemma-disjoint)", color="#DD8452")
    ax.set_xticks(xpos); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(metric_key); ax.set_ylim(0, 1); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, dpi=FIG_DPI); plt.close(fig)


grouped("accuracy", "accuracy_A_vs_B.png")
grouped("macro_f1", "macro_f1_A_vs_B.png")


def gap_ci(metric_name, delta_key, ci_key, fname):
    """One figure per metric, faceted by task; zero stays inside every panel."""
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.0))
    for ax, task in zip(axes, ("upos", "number")):
        d = np.array([gap_report[(c, task)][delta_key] for c in LANGS])
        lo = np.array([gap_report[(c, task)][ci_key][0] for c in LANGS])
        hi = np.array([gap_report[(c, task)][ci_key][1] for c in LANGS])
        xp = np.arange(len(LANGS))
        ax.errorbar(xp, d, yerr=[d - lo, hi - d], fmt="o", capsize=4, color="#C44E52")
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.set_xticks(xp); ax.set_xticklabels(LANGS, fontsize=8)
        ax.set_xlim(-0.5, len(LANGS) - 0.5)
        ax.set_ylim(-0.08 * hi.max(), 1.12 * hi.max())
        ax.set_ylabel(f"delta {metric_name} (A - B)")
        ax.set_title(TASK_TITLE[task], fontsize=10)
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, dpi=FIG_DPI); plt.close(fig)


gap_ci("accuracy", "delta_acc", "acc_ci", "gap_ci_accuracy.png")
gap_ci("macro-F1", "delta_f1", "f1_ci", "gap_ci_macro_f1.png")


def selectivity(metric_key, metric_name, fname):
    """Task metric minus control metric, per condition; one figure per metric."""
    selA = [main_report[(c, t)]["conditions"]["A"]["probe"][metric_key]
            - control_report[(c, t)]["conditions"]["A"]["control"][metric_key] for (c, t) in cells]
    selB = [main_report[(c, t)]["conditions"]["B"]["probe"][metric_key]
            - control_report[(c, t)]["conditions"]["B"]["control"][metric_key] for (c, t) in cells]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(xpos - 0.2, selA, 0.4, label="sel_A", color="#55A868")
    ax.bar(xpos + 0.2, selB, 0.4, label="sel_B", color="#8172B3")
    ax.set_xticks(xpos); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(f"selectivity ({metric_name})"); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / fname, dpi=FIG_DPI); plt.close(fig)


selectivity("accuracy", "accuracy", "selectivity_accuracy.png")
selectivity("macro_f1", "macro-F1", "selectivity_macro_f1.png")

figs = ["accuracy_A_vs_B.png", "macro_f1_A_vs_B.png", "gap_ci_accuracy.png",
        "gap_ci_macro_f1.png", "selectivity_accuracy.png", "selectivity_macro_f1.png"]
print("wrote " + ", ".join(f"results/figures/{f}" for f in figs)
      + f"   ({FIG_DPI} dpi)")


wrote results/figures/accuracy_A_vs_B.png, results/figures/macro_f1_A_vs_B.png, results/figures/gap_ci_accuracy.png, results/figures/gap_ci_macro_f1.png, results/figures/selectivity_accuracy.png, results/figures/selectivity_macro_f1.png   (300 dpi)


### Thesis Body Figures

Draws body figures. Writes `fig3_1_split_construction.png`, `fig5_1_per_class_f1_drop.png`, and `fig5_2_task_vs_control.png`.

In [34]:
# thesis body figures, from frozen files in results/; runs on its own after the path cells
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

BLUE, BLUE_LT, ORANGE, ORANGE_LT = "#2a78d6", "#b7d3f6", "#eb6834", "#f7c3ad"
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
PLACEHOLDER = "#f3f2ef"
TEXT_W = 16.5 / 2.54
THESIS_STYLE = {
    "font.size": 8, "axes.titlesize": 8.5,
    "axes.labelsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.dpi": 300,
}
TB_NAME = {"en_ewt": "EN-EWT", "it_isdt": "IT-ISDT", "pl_pdb": "PL-PDB"}
THESIS_FIGS = ["fig3_1_split_construction.png", "fig5_1_per_class_f1_drop.png",
               "fig5_2_task_vs_control.png"]


def declutter(ax):
    """Open axes (no top/right spines) over a light horizontal grid."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)


def fig_split_construction(path):
    """Figure 3.1: the blocks T, S, R and K of one cell, to scale."""
    sp = json.loads((RESULTS_DIR / "split_report.json").read_text(encoding="utf-8"))["en_ewt/upos"]
    w = {k: 100 * sp[k] / sp["N"] for k in ("T", "S", "R", "K")}   # % of eligible occurrences
    x0 = {"T": 0.0}
    x0["S"] = x0["T"] + w["T"]
    x0["R"] = x0["S"] + w["S"]
    x0["K"] = x0["R"] + w["R"]
    fills = {"T": (ORANGE, "white"), "S": (ORANGE_LT, INK), "R": (BLUE_LT, INK), "K": (BLUE, "white")}
    names = {"T": "Test", "S": "Seen", "R": "Reserve", "K": "Core"}
    row_h, gap = 0.62, 0.35   # white seam between blocks
    y_all, y_a, y_b = 2.55, 1.45, 0.4

    fig, ax = plt.subplots(figsize=(TEXT_W, 1.9))
    ax.set_xlim(-27, 101)
    ax.set_ylim(0.15, 3.8)
    ax.axis("off")

    def block(key, y, text=None):
        face, tcolor = fills[key]
        ax.add_patch(Rectangle((x0[key] + gap / 2, y), w[key] - gap, row_h, facecolor=face,
                               edgecolor="none"))
        if text:
            ax.text(x0[key] + w[key] / 2, y + row_h / 2, text, ha="center", va="center",
                    color=tcolor, fontsize=7.5)

    def unused(key, y, text=""):   # block not in this condition's training
        ax.add_patch(Rectangle((x0[key] + gap / 2, y), w[key] - gap, row_h, facecolor=PLACEHOLDER,
                               edgecolor="none"))
        if text:
            ax.text(x0[key] + w[key] / 2, y + row_h / 2, text, ha="center", va="center",
                    color=MUTED, fontsize=7, style="italic")

    def bracket(xa, xb, y, label):
        ax.plot([xa + 0.3, xa + 0.3, xb - 0.3, xb - 0.3], [y, y + 0.14, y + 0.14, y],
                color=INK2, lw=0.8, solid_capstyle="butt")
        ax.text((xa + xb) / 2, y + 0.24, label, ha="center", va="bottom", fontsize=7.5, color=INK)

    for k in ("T", "S", "R", "K"):
        block(k, y_all, f"{names[k]} ({k})")
    bracket(x0["T"], x0["S"] + w["S"], y_all + row_h + 0.08, "Held-out lemmata (H)")
    bracket(x0["R"], x0["K"] + w["K"], y_all + row_h + 0.08, "Background pool (P)")
    unused("T", y_a, "Tested on T")   # condition A trains on K u S
    block("S", y_a, "S")
    unused("R", y_a)
    block("K", y_a, "K")
    unused("T", y_b, "Tested on T")   # condition B trains on K u R
    unused("S", y_b)
    block("R", y_b, "R")
    block("K", y_b, "K")
    for y, text in ((y_all, "All eligible\noccurrences"), (y_a, "Lexically overlapping (A)"),
                    (y_b, "Lemma-disjoint (B)")):
        ax.text(-1.5, y + row_h / 2, text, ha="right", va="center", fontsize=7.5)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
    fig.savefig(path)
    plt.close(fig)


def fig_per_class_f1_drop(path):
    """Figure 5.1: per-class UPOS F1 drop against class support in T."""
    pc = pd.read_csv(RESULTS_DIR / "per_class_f1.csv")
    pc = pc[pc["task"] == "upos"]
    chk = pd.read_csv(RESULTS_DIR / "per_class_f1_support_check.csv").set_index(["language", "task"])
    colour = {"closed": ORANGE, "open": BLUE, "other": MUTED}

    fig, axes = plt.subplots(1, 3, figsize=(TEXT_W, 2.45), sharey=True)
    for ax, code in zip(axes, TB_NAME):
        t = pc[pc["language"] == code]
        for ctype in ("open", "closed", "other"):
            u = t[t["class_type"] == ctype]
            ax.scatter(u["support_T"].clip(lower=1), u["delta_f1"], s=22, color=colour[ctype],
                       edgecolor="white", linewidth=0.6, zorder=3,
                       label=f"{ctype} class" if ctype != "other" else "other (PUNCT, SYM, X)")
        rho = chk.loc[(code, "upos"), "spearman_support_delta"]
        ax.set_xscale("log")
        ax.set_xlim(1, 6e4)
        ax.set_ylim(-0.06, 1.08)
        ax.axhline(0, color=AXIS, lw=0.8, zorder=1)
        ax.set_xlabel("class support in T (log scale)")
        ax.set_title(f"{TB_NAME[code]}   ρ = " + f"{rho:+.2f}".replace("-", "−"), color=INK)
        declutter(ax)
    axes[0].set_ylabel("ΔF1 = F1(A) − F1(B)")
    axes[0].legend(loc="upper left", frameon=False, handletextpad=0.2, borderaxespad=0.1)
    fig.subplots_adjust(left=0.07, right=0.995, top=0.91, bottom=0.18, wspace=0.07)

    # label large drops and rare classes only; offsets set manually to avoid collisions
    right, left, above = (4, 0, "left", "center"), (-4, 0, "right", "center"), (0, 4, "center", "bottom")
    offsets = {"en_ewt": {"SCONJ": left, "CCONJ": left, "NUM": left},
               "it_isdt": {"PUNCT": (-4, 1.5, "right", "center")},
               "pl_pdb": {"PART": above, "CCONJ": left, "ADV": (4, 1.5, "left", "center"), "SCONJ": left}}
    for ax, code in zip(axes, TB_NAME):
        for r in pc[pc["language"] == code].itertuples():
            if r.delta_f1 < 0.15 and r.support_T >= 100:
                continue
            dx, dy, ha, va = offsets[code].get(r.label, right)
            ax.annotate(r.label, (max(r.support_T, 1), r.delta_f1), xytext=(dx, dy),
                        textcoords="offset points", ha=ha, va=va, fontsize=6, color=INK2, zorder=4)
    fig.savefig(path)
    plt.close(fig)


def fig_task_vs_control(path):
    """Figure 5.2: task and control accuracy from condition A to B."""
    mr = pd.read_csv(RESULTS_DIR / "main_results.csv").set_index(["language", "task", "condition"])

    fig, axes = plt.subplots(1, 2, figsize=(TEXT_W, 2.3), sharey=True)
    for ax, task, title in zip(axes, ("upos", "number"), ("UPOS", "Number")):
        ticks, ticklabels = [], []
        for i, code in enumerate(TB_NAME):
            xa, xb = i * 2.3, i * 2.3 + 1
            a, b = mr.loc[(code, task, "A")], mr.loc[(code, task, "B")]
            floor = a["control_majority_acc"]
            ax.plot([xa - 0.3, xb + 0.3], [floor, floor], color=MUTED, lw=1.0,
                    label="control majority floor" if i == 0 else None, zorder=1)
            ax.plot([xa, xb], [a["accuracy"], b["accuracy"]], color=BLUE, lw=2, marker="o", ms=5,
                    mec="white", mew=1, label="task probe" if i == 0 else None, zorder=3)
            ax.plot([xa, xb], [a["control_acc"], b["control_acc"]], color=ORANGE, lw=2, marker="o",
                    ms=5, mec="white", mew=1, label="control probe" if i == 0 else None, zorder=3)
            ticks += [xa, xb]
            ticklabels += ["A", "B"]
            ax.text((xa + xb) / 2, -0.13, TB_NAME[code], ha="center", va="top", fontsize=7.5,
                    transform=ax.get_xaxis_transform())
        ax.set_xticks(ticks)
        ax.set_xticklabels(ticklabels)
        ax.tick_params(axis="x", length=0)
        ax.set_xlim(-0.6, 2 * 2.3 + 1.6)
        ax.set_ylim(0, 1.02)
        ax.set_title(title, color=INK)
        declutter(ax)
    axes[0].set_ylabel("accuracy on T")
    handles, labels = axes[0].get_legend_handles_labels()
    order = [1, 2, 0]   # task, control, then floor
    axes[1].legend([handles[i] for i in order], [labels[i] for i in order], loc="lower center",
                   bbox_to_anchor=(0.5, 0.06), frameon=False, handlelength=1.8)
    fig.subplots_adjust(left=0.075, right=0.995, top=0.9, bottom=0.17, wspace=0.08)
    fig.savefig(path)
    plt.close(fig)


with plt.rc_context(THESIS_STYLE):
    for draw, fname in zip((fig_split_construction, fig_per_class_f1_drop, fig_task_vs_control),
                           THESIS_FIGS):
        draw(FIG_DIR / fname)
print("wrote " + ", ".join(f"results/figures/{f}" for f in THESIS_FIGS)
      + "   (Figures 3.1, 5.1, 5.2; 16.5 cm wide, 300 dpi)")

wrote results/figures/fig3_1_split_construction.png, results/figures/fig5_1_per_class_f1_drop.png, results/figures/fig5_2_task_vs_control.png   (Figures 3.1, 5.1, 5.2; 16.5 cm wide, 300 dpi)


In [35]:
# POSIX paths; vectors are the one artifact a clone does not carry
def rel(p):
    return p.relative_to(PROJECT).as_posix()


artifacts = {
    "config": "config.json",
    "run_manifest": "results/run_manifest.json",
    "reports_json": sorted(rel(p) for p in RESULTS_DIR.glob("*.json")),
    "tables_csv": sorted(rel(p) for p in RESULTS_DIR.glob("*.csv")),
    "figures": sorted(rel(p) for p in FIG_DIR.glob("*.png")),
    "representation_cache": {
        "in_repository": sorted(rel(p) for p in [*REPR_DIR.glob("*_meta.parquet"), *REPR_DIR.glob("*.marker.json")]),
        "local_only": sorted(rel(p) for p in REPR_DIR.glob("*_layer*.npy")),
    },
    "frozen_splits": sorted(rel(p) for p in SPLIT_DIR.glob("*_*.parquet")),
    "cached_predictions": sorted(rel(p) for p in PRED_DIR.glob("*.parquet")),
}
(RESULTS_DIR / "artifacts_index.json").write_text(json.dumps(artifacts, indent=2), encoding="utf-8")
print("wrote results/artifacts_index.json")

wrote results/artifacts_index.json


## 10. Robustness Extensions

One MLP width (256 units) on layer 8. Layers 4, 8, and 11 probed independently. Writes `results/extension_mlp_probe.csv`, `extension_layer_sweep.csv`, and `extension_summary.csv`.

In [36]:
# one MLP width
from sklearn.neural_network import MLPClassifier

EXT_UNITS = CONFIG["extensions"]["mlp_hidden_units"]
d1_path = RESULTS_DIR / "extension_mlp_probe.csv"


def ext_fit_eval(make_estimator, code, task, layer, probe_name, path):
    """Fit both conditions under the probe pipeline on a given layer; append one row."""
    # layer 8 is extracted here if missing; D.2 extracts the other depths
    X = (load_vectors(code) if layer == CONFIG["layer_index"]
         else np.load(REPR_DIR / f"{code}_layer{layer}.npy", mmap_mode="r"))
    d = pd.read_parquet(SPLIT_DIR / f"{code}_{task}.parquet")
    part, repr_row = d["partition"].to_numpy(), d["repr_row"].to_numpy()
    y = d["gold_label"].to_numpy()
    test = d[part == "test"].sort_values("repr_row")
    Xte, yte = np.asarray(X[test["repr_row"].to_numpy()]), test["gold_label"].to_numpy()
    r = {}
    for cond, keep in (("A", ["core", "seen"]), ("B", ["core", "reserve"])):
        mask = np.isin(part, keep)
        t0 = time.time()
        pipe = make_pipeline(StandardScaler(), make_estimator())
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            pipe.fit(np.asarray(X[repr_row[mask]]), y[mask])
        pred = pipe.predict(Xte)
        r[cond] = {**point_metrics(yte, pred), "n_train": int(mask.sum()),
                   "n_iter": int(np.max(pipe[-1].n_iter_)), "fit_s": round(time.time() - t0, 1),
                   "converged": not any(issubclass(w.category, ConvergenceWarning) for w in caught)}
        del pipe
        gc.collect()
    row = {"language": code, "task": task, "probe": probe_name, "layer": layer,
           "acc_A": r["A"]["accuracy"], "acc_B": r["B"]["accuracy"],
           "f1_A": r["A"]["macro_f1"], "f1_B": r["B"]["macro_f1"],
           "delta_acc": r["A"]["accuracy"] - r["B"]["accuracy"],
           "delta_f1": r["A"]["macro_f1"] - r["B"]["macro_f1"],
           "n_train": r["A"]["n_train"], "n_test": len(yte),
           "n_iter_A": r["A"]["n_iter"], "n_iter_B": r["B"]["n_iter"],
           "converged_A": r["A"]["converged"], "converged_B": r["B"]["converged"],
           "fit_s_A": r["A"]["fit_s"], "fit_s_B": r["B"]["fit_s"]}
    pd.DataFrame([row]).to_csv(path, mode="a", header=not path.exists(), index=False)
    del X, Xte
    gc.collect()
    return row


def ext_done(path, keys):
    return set() if not path.exists() else set(map(tuple, pd.read_csv(path)[keys].to_numpy().tolist()))


have = ext_done(d1_path, ["language", "task"])
for (code, task) in main_report:
    if (code, task) in have:
        continue
    print(f"  fitting MLP-{EXT_UNITS}: {code}/{task} (several minutes)")
    ext_fit_eval(lambda: MLPClassifier(hidden_layer_sizes=(EXT_UNITS,), max_iter=200,
                                       random_state=SEED, early_stopping=False),
                 code, task, CONFIG["layer_index"], f"mlp_{EXT_UNITS}", d1_path)

d1 = pd.read_csv(d1_path)
print(f"D.1 — MLP with {EXT_UNITS} hidden units on layer {CONFIG['layer_index']}, "
      f"all converged={bool(d1.converged_A.all() and d1.converged_B.all())} "
      f"({d1.n_iter_A.min()}-{d1.n_iter_A.max()} epochs)")
print(d1[["language", "task", "f1_A", "f1_B", "delta_f1", "delta_acc"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

D.1 — MLP with 256 hidden units on layer 8, all converged=True (17-55 epochs)
language   task   f1_A   f1_B  delta_f1  delta_acc
  en_ewt   upos 0.9326 0.6815    0.2512     0.0951
  en_ewt number 0.9935 0.9920    0.0015     0.0011
 it_isdt   upos 0.8428 0.6586    0.1842     0.1735
 it_isdt number 0.9970 0.9962    0.0008     0.0007
  pl_pdb   upos 0.9125 0.7086    0.2039     0.1084
  pl_pdb number 0.9900 0.9859    0.0040     0.0032


In [37]:
# rows keep layer-8 metadata order, so repr_row addresses same target at every depth
d2_path = RESULTS_DIR / "extension_layer_sweep.csv"
SWEEP = CONFIG["extensions"]["layer_sweep"]
extra = [L for L in SWEEP if L != CONFIG["layer_index"]]


def extract_extra_layers(code, layers):
    """Re-run the Part 4 alignment, keeping several depths."""
    todo = [L for L in layers if not (REPR_DIR / f"{code}_layer{L}.npy").exists()]
    if not todo:
        return
    cols = ["treebank", "native_split", "sent_id", "word_id", "form", "lemma", "upos",
            "number", "char_length"]
    union = (pd.concat([inventories[(code, "upos")][cols], inventories[(code, "number")][cols]])
             .drop_duplicates(["sent_id", "word_id"]))
    wids_by_sid = union.groupby("sent_id")["word_id"].agg(list).to_dict()
    sents = []
    for _, sent in iter_pooled_sentences(code):
        wids = wids_by_sid.get(sent.metadata["sent_id"])
        if not wids:
            continue
        text, spans = reconstruct_sentence(sent)
        sents.append((sent.metadata["sent_id"], text, [(w, *spans[w]) for w in wids]))
    texts = [t for _, t, _ in sents]
    lens = [len(i) for i in tokenizer(texts, add_special_tokens=True, truncation=True,
                                      max_length=CONFIG["max_length"])["input_ids"]]
    batches = pack_batches(sorted(range(len(sents)), key=lambda i: -lens[i]), lens)
    meta = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet")
    row_of = {(r.sent_id, r.word_id): r.row for r in meta.itertuples(index=False)}
    Xs = {L: np.zeros((len(meta), 768), dtype=np.float32) for L in todo}
    filled = np.zeros(len(meta), dtype=bool)
    t0 = time.time()
    for bi, batch in enumerate(batches):
        enc = tokenizer([texts[i] for i in batch], padding=True, truncation=True,
                        max_length=CONFIG["max_length"], add_special_tokens=True,
                        return_offsets_mapping=True, return_tensors="pt")
        with torch.no_grad():
            out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        for j, i_sent in enumerate(batch):
            sid, _, tgts = sents[i_sent]
            offs = enc["offset_mapping"][j].tolist()
            for wid, s, e in tgts:
                idxs = subwords_for_span(offs, s, e)
                r = row_of.get((sid, wid))
                if not idxs or r is None:
                    continue
                for L in todo:
                    Xs[L][r] = out.hidden_states[L][j, idxs[-1]].numpy()
                filled[r] = True
        if (bi + 1) % 50 == 0:
            print(f"    {code}: batch {bi + 1}/{len(batches)} ({time.time() - t0:,.0f}s)")
        del enc, out
    assert filled.all(), f"{code}: {(~filled).sum()} rows unfilled — alignment diverged"
    for L in todo:
        np.save(REPR_DIR / f"{code}_layer{L}.npy", Xs[L])
    print(f"  {code}: layers {todo} extracted in {time.time() - t0:,.0f}s")
    del Xs
    gc.collect()


have = ext_done(d2_path, ["language", "task", "layer"])
d2_todo = [(L, code, task) for L in extra for (code, task) in main_report if (code, task, L) not in have]
for code in CONFIG["treebanks"].values():
    if any(c == code for _, c, _ in d2_todo):   # the extra depths only where a fit remains
        extract_extra_layers(code, extra)
for L, code, task in d2_todo:
    print(f"  fitting layer {L}: {code}/{task}")
    ext_fit_eval(lambda: LogisticRegression(**PROBE_KW), code, task, L, "logreg_C1", d2_path)

d2 = pd.read_csv(d2_path)
print(f"D.2 — layers {SWEEP} probed independently ({CONFIG['layer_index']} is the primary), "
      f"all converged={bool(d2.converged_A.all() and d2.converged_B.all())}")
print(d2[["language", "task", "layer", "f1_A", "f1_B", "delta_f1", "delta_acc"]]
      .sort_values(["language", "task", "layer"])
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

D.2 — layers [4, 8, 11] probed independently (8 is the primary), all converged=True
language   task  layer   f1_A   f1_B  delta_f1  delta_acc
  en_ewt number      4 0.9906 0.9836    0.0070     0.0051
  en_ewt number     11 0.9862 0.9751    0.0112     0.0080
  en_ewt   upos      4 0.9160 0.6052    0.3109     0.1401
  en_ewt   upos     11 0.9145 0.6676    0.2468     0.0887
 it_isdt number      4 0.9943 0.9922    0.0021     0.0018
 it_isdt number     11 0.9926 0.9912    0.0015     0.0013
 it_isdt   upos      4 0.8459 0.5676    0.2784     0.2419
 it_isdt   upos     11 0.8628 0.6314    0.2315     0.1321
  pl_pdb number      4 0.9648 0.9562    0.0086     0.0069
  pl_pdb number     11 0.9861 0.9806    0.0055     0.0044
  pl_pdb   upos      4 0.8967 0.6182    0.2785     0.1296
  pl_pdb   upos     11 0.8440 0.6091    0.2349     0.1337


In [38]:
# checks against the primary probe
ext = pd.concat([pd.read_csv(RESULTS_DIR / f) for f in
                 ("extension_mlp_probe.csv", "extension_layer_sweep.csv")], ignore_index=True)
base = {(c, t): {"f1_A": s["conditions"]["A"]["probe"]["macro_f1"],
                 "f1_B": s["conditions"]["B"]["probe"]["macro_f1"]}
        for (c, t), s in main_report.items()}
ext["base_f1_B"] = [base[(r.language, r.task)]["f1_B"] for r in ext.itertuples()]
ext["base_delta_f1"] = [base[(r.language, r.task)]["f1_A"] - base[(r.language, r.task)]["f1_B"]
                        for r in ext.itertuples()]
ext["f1_B_vs_base"] = ext["f1_B"] - ext["base_f1_B"]
ext["delta_f1_vs_base"] = ext["delta_f1"] - ext["base_delta_f1"]
ext.to_csv(RESULTS_DIR / "extension_summary.csv", index=False)

# direction must hold in every extension cell
assert (ext["delta_f1"] > 0).all() and (ext["delta_acc"] > 0).all()
print(f"A > B on both metrics in all {len(ext)} extension cells "
      f"(min macro-F1 gap {ext.delta_f1.min():.4f})\n")

for task in ("upos", "number"):
    print(f"{task}: lemma-disjoint macro-F1 by variant (primary in brackets)")
    for code in CONFIG["treebanks"].values():
        sub = ext[(ext.language == code) & (ext.task == task)].sort_values(["probe", "layer"])
        cells = "  ".join(f"{r.probe.replace('logreg_C1', 'L')}{'' if r.probe.startswith('mlp') else r.layer}"
                          f"={r.f1_B:.3f}" for r in sub.itertuples())
        print(f"  {code:8s} [{base[(code, task)]['f1_B']:.3f}]  {cells}")
    print()

worst = ext.loc[ext["delta_f1_vs_base"].idxmin()]
print(f"largest narrowing of the macro-F1 gap: {worst.language}/{worst.task} under {worst.probe} "
      f"L{worst.layer}, {worst.base_delta_f1:.3f} -> {worst.delta_f1:.3f} "
      f"({worst.delta_f1_vs_base:+.3f})")

# refreshed here: Part 10 wrote tables after the index was built
artifacts["tables_csv"] = sorted(rel(p) for p in RESULTS_DIR.glob("*.csv"))
(RESULTS_DIR / "artifacts_index.json").write_text(json.dumps(artifacts, indent=2), encoding="utf-8")
print(f"\nwrote results/extension_summary.csv; artifact index refreshed "
      f"({len(artifacts['tables_csv'])} CSV tables)")

A > B on both metrics in all 18 extension cells (min macro-F1 gap 0.0008)

upos: lemma-disjoint macro-F1 by variant (primary in brackets)
  en_ewt   [0.677]  L4=0.605  L11=0.668  mlp_256=0.681
  it_isdt  [0.683]  L4=0.568  L11=0.631  mlp_256=0.659
  pl_pdb   [0.622]  L4=0.618  L11=0.609  mlp_256=0.709

number: lemma-disjoint macro-F1 by variant (primary in brackets)
  en_ewt   [0.981]  L4=0.984  L11=0.975  mlp_256=0.992
  it_isdt  [0.990]  L4=0.992  L11=0.991  mlp_256=0.996
  pl_pdb   [0.982]  L4=0.956  L11=0.981  mlp_256=0.986

largest narrowing of the macro-F1 gap: pl_pdb/upos under mlp_256 L8, 0.283 -> 0.204 (-0.079)

wrote results/extension_summary.csv; artifact index refreshed (18 CSV tables)


## 11. Additional Statistics Quoted in the Body

Writes `results/error_analysis_standardization.csv`, `error_analysis_model_size.csv`, and `text_statistics.csv`, one row per statistic and cell, tagged with the thesis section that quotes it.

In [39]:
# rebuilds the Part 8 regression frame from the cached predictions, without refitting
TB_CODES = ("en_ewt", "it_isdt", "pl_pdb")
PREDICTORS = ("n_subwords", "char_length", "log_freq")
B3_ORDER = [(c, t) for c in (*TB_CODES, "pooled") for t in ("upos", "number")]   # thesis row order

frag_frame = {}
for code in TB_CODES:
    meta = pd.read_parquet(REPR_DIR / f"{code}_meta.parquet",
                           columns=["sent_id", "word_id", "form", "n_subwords", "char_length"])
    form_count = meta["form"].value_counts()
    for task in ("upos", "number"):
        df = pd.read_parquet(PRED_DIR / f"{code}_{task}.parquet").merge(
            meta, on=["sent_id", "word_id"], how="left")
        df["error"] = (df["pred_B"].to_numpy() != df["gold_label"].to_numpy()).astype(int)
        df["log_freq"] = np.log(df["form"].map(form_count).fillna(0).to_numpy() + 1.0)
        frag_frame[(code, task)] = df

coef_tab = pd.concat([pd.read_csv(RESULTS_DIR / "error_analysis_coefficients.csv"),
                      pd.read_csv(RESULTS_DIR / "error_analysis_pooled.csv").assign(language="pooled")],
                     ignore_index=True)
std_rows, size_rows = [], []
for code, task in B3_ORDER:
    df = (pd.concat([frag_frame[(c, task)] for c in TB_CODES]) if code == "pooled"
          else frag_frame[(code, task)])   # pooled: re-standardized
    for p in PREDICTORS:
        std_rows.append({"language": code, "task": task, "predictor": p,
                         "raw_mean": float(df[p].mean()), "raw_sd": float(df[p].std(ddof=0))})
    n_err = int(df["error"].sum())
    n_par = int(((coef_tab["language"] == code) & (coef_tab["task"] == task)).sum())
    size_rows.append({"language": code, "task": task, "test_occurrences": len(df), "errors": n_err,
                      "error_rate": n_err / len(df), "parameters": n_par,
                      "events_per_parameter": n_err / n_par})

frag_std, frag_size = pd.DataFrame(std_rows), pd.DataFrame(size_rows)
frag_std.to_csv(RESULTS_DIR / "error_analysis_standardization.csv", index=False)
frag_size.to_csv(RESULTS_DIR / "error_analysis_model_size.csv", index=False)

print("Table B.3.1 -- raw mean / SD behind each z-scored predictor")
print(frag_std.assign(mean_sd=[f"{m:.3f} / {s:.3f}" for m, s in zip(frag_std.raw_mean, frag_std.raw_sd)])
      .set_index(["language", "task", "predictor"])["mean_sd"].unstack("predictor")
      .reindex(index=pd.MultiIndex.from_tuples(B3_ORDER, names=["language", "task"]),
               columns=list(PREDICTORS)).to_string())
print("\nTable B.3.2 -- fitted model sizes and events per estimated parameter")
print(frag_size.to_string(index=False, formatters={"error_rate": "{:.2%}".format,
                                                   "events_per_parameter": "{:.1f}".format}))
print("\nwrote results/error_analysis_standardization.csv (Table B.3.1), "
      "results/error_analysis_model_size.csv (Table B.3.2)")

Table B.3.1 -- raw mean / SD behind each z-scored predictor
predictor           n_subwords    char_length       log_freq
language task                                               
en_ewt   upos    1.190 / 0.548  4.424 / 2.675  4.804 / 2.301
         number  1.363 / 0.653  6.091 / 2.416  2.847 / 1.192
it_isdt  upos    1.225 / 0.524  3.837 / 3.092  6.100 / 2.749
         number  1.500 / 0.721  7.423 / 2.487  3.182 / 1.209
pl_pdb   upos    1.563 / 0.874  5.321 / 3.215  4.406 / 2.777
         number  1.957 / 0.966  7.336 / 2.466  2.539 / 1.337
pooled   upos    1.356 / 0.719  4.625 / 3.098  5.022 / 2.738
         number  1.682 / 0.872  7.073 / 2.520  2.803 / 1.296

Table B.3.2 -- fitted model sizes and events per estimated parameter
language   task  test_occurrences  errors error_rate  parameters events_per_parameter
  en_ewt   upos             49630    6647     13.39%          20                332.4
  en_ewt number              8510     115      1.35%           5                 23.0
 i

In [40]:
# one row per statistic and cell; shares as fractions, pp in points, flag = 1 where it holds
import hashlib
import json
import unicodedata

from transformers import AutoTokenizer

missing = [f for f in ("iter_pooled_sentences", "reconstruct_sentence") if f not in globals()]
assert not missing, f"run the CoNLL-U parsing cells of Parts 3 and 4 first (they define {missing})"

frozen_cfg = json.loads((PROJECT / "config.json").read_text(encoding="utf-8"))
run_manifest = json.loads((RESULTS_DIR / "run_manifest.json").read_text(encoding="utf-8"))
TB_CODES = tuple(frozen_cfg["treebanks"].values())
LT_CELLS = [(c, t) for c in TB_CODES for t in ("upos", "number")]
CLOSED_TAGS = {"ADP", "AUX", "CCONJ", "DET", "PART", "PRON", "SCONJ"}   # NUM open, as throughout
SECTIONS = ["Linguistic Preliminaries", "Multilingual Encoders and XLM-R", "Tasks and Data",
            "Control Task and Selectivity", "Representation Extraction", "Data Volumes",
            "Primary Result: Seen/Unseen Gap", "Selectivity", "Fragmentation",
            "Number Scores Near Ceiling", "Inconclusive Fragmentation Result", "Threats to Validity"]
text_stats = []


def stat(section, statistic, value, language="all", task="all", unit="value", note=""):
    assert section in SECTIONS, section
    text_stats.append({"section": section, "statistic": statistic, "language": language, "task": task,
                       "value": float(value), "unit": unit, "note": note})


inv_report = json.loads((RESULTS_DIR / "inventory_report.json").read_text(encoding="utf-8"))
ext_report = json.loads((RESULTS_DIR / "extraction_report.json").read_text(encoding="utf-8"))
split_rep = json.loads((RESULTS_DIR / "split_report.json").read_text(encoding="utf-8"))
main_res = pd.read_csv(RESULTS_DIR / "main_results.csv").set_index(["language", "task", "condition"])
gap_tab = pd.read_csv(RESULTS_DIR / "gap_table.csv").set_index(["language", "task"])
mcn_tab = pd.read_csv(RESULTS_DIR / "mcnemar_table.csv").set_index(["language", "task"])
pcf = pd.read_csv(RESULTS_DIR / "per_class_f1.csv")
coef = pd.read_csv(RESULTS_DIR / "error_analysis_coefficients.csv")

# a lemma type is closed-class if it ever takes a closed tag
for code in TB_CODES:
    inv = pd.read_parquet(CACHE_DIR / "inventory" / f"{code}_upos.parquet", columns=["lemma", "upos"])
    closed = inv["upos"].isin(CLOSED_TAGS)
    stat("Linguistic Preliminaries", "closed-class lemma types / lemma types",
         inv.loc[closed, "lemma"].nunique() / inv["lemma"].nunique(), code, "upos", "share")
    stat("Linguistic Preliminaries", "closed-class tokens / tokens", closed.mean(), code, "upos", "share")

# casing changes the segmentation
tok = AutoTokenizer.from_pretrained(run_manifest["model_name"], revision=run_manifest["model_revision"])
for word in ("Kraków", "kraków"):
    pieces = tok.tokenize(word)
    stat("Multilingual Encoders and XLM-R", f"subwords of '{word}'", len(pieces), unit="count",
         note=" ".join(pieces))

# skew of the two label distributions
for code in TB_CODES:
    support = pd.Series(inv_report[code]["upos"]["per_class_support"]).sort_values(ascending=False)
    stat("Tasks and Data", "three most frequent UPOS tags / tokens", support.iloc[:3].sum() / support.sum(),
         code, "upos", "share", ", ".join(support.index[:3]))
    stat("Tasks and Data", "tokens of the least frequent UPOS tag", support.iloc[-1], code, "upos",
         "count", support.index[-1])
    number = inv_report[code]["number"]["per_class_support"]
    stat("Tasks and Data", "Sing / Number targets", number["Sing"] / sum(number.values()), code,
         "number", "share")

# lemma types that received a control label
for code, task in LT_CELLS:
    stat("Control Task and Selectivity", "lemma types with a control label",
         len(pd.read_parquet(CACHE_DIR / "control" / f"{code}_{task}.parquet")), code, task, "count")

# rebuilt text vs the '# text' metadata, and whether a difference hits a target
for code in TB_CODES:
    targets = (pd.read_parquet(CACHE_DIR / "inventory" / f"{code}_upos.parquet", columns=["sent_id", "word_id"])
               .groupby("sent_id")["word_id"].agg(set).to_dict())
    n_diff, n_hit, kinds = 0, 0, set()
    for _, sent in iter_pooled_sentences(code):
        sid, meta_text = sent.metadata["sent_id"], sent.metadata.get("text")
        if sid not in targets:
            continue
        text, spans = reconstruct_sentence(sent)
        if meta_text in (None, text):
            continue
        n_diff += 1
        pos = [i for i in range(max(len(text), len(meta_text)))
               if i >= min(len(text), len(meta_text)) or text[i] != meta_text[i]]
        n_hit += sum(any(s <= i < e for i in pos) for w, (s, e) in spans.items() if w in targets[sid])
        kinds |= {f"{unicodedata.name(meta_text[i], '?')} in # text vs {unicodedata.name(text[i], '?')}"
                  for i in pos if i < min(len(text), len(meta_text))}
    assert n_diff == ext_report[code]["text_metadata_mismatches"], code   # agrees with the extraction report
    stat("Representation Extraction", "sentences whose rebuilt text differs from # text", n_diff, code,
         unit="count", note="; ".join(sorted(kinds)))
    stat("Representation Extraction", "targets whose span covers a differing character", n_hit, code,
         unit="count")

# extraction as a whole
n_vectors = sum(ext_report[c]["n_rows"] for c in TB_CODES)
stat("Data Volumes", "target vectors extracted", n_vectors, unit="count")
stat("Data Volumes", "layer-8 representation cache (GB)", n_vectors * 768 * 4 / 1e9, unit="GB",
     note="float32 vectors of 768 dimensions")
for code in TB_CODES:
    stat("Data Volumes", "target vectors / all target vectors", ext_report[code]["n_rows"] / n_vectors,
         code, unit="share")
    stat("Data Volumes", "Number targets / eligible tokens",
         inv_report[code]["number"]["n_targets"] / inv_report[code]["eligible_tokens"], code, "number", "share")

# structure of each split
split_cfg = frozen_cfg["split"]
for code, task in LT_CELLS:
    d = pd.read_parquet(SPLIT_DIR / f"{code}_{task}.parquet")
    s = split_rep[f"{code}/{task}"]
    N, part, lemma, gold = len(d), d["partition"], d["lemma"], d["gold_label"]
    per_lemma = d.groupby("lemma").size()   # sorted by lemma, as in the split cell
    count_of = per_lemma.to_dict()
    single = per_lemma == 1
    held_out = set(lemma[part.isin(["test", "seen"])])
    test_per_h = d[part == "test"].groupby("lemma").size().sort_values(ascending=False)
    sec = "Data Volumes"
    stat(sec, "|P| = N - (|T| + s)", s["P"] == N - (s["T"] + s["S"]), code, task, "flag")
    stat(sec, "held-out occurrences (|T| + s) / N", (s["T"] + s["S"]) / N, code, task, "share")
    stat(sec, "|P| / s", s["P"] / s["S"], code, task, "ratio")
    stat(sec, "|T| / N", s["T"] / N, code, task, "share")
    stat(sec, "single-occurrence lemmata / lemma types", single.mean(), code, task, "share",
         "also the Introduction's 'around half' and Threats to Validity")
    stat(sec, "single-occurrence lemmata / tokens", single.sum() / N, code, task, "share")
    stat(sec, "|H| / lemma types with two or more occurrences", len(held_out) / (~single).sum(), code,
         task, "share")
    stat(sec, "test occurrences per held-out lemma, mean", test_per_h.mean(), code, task,
         note="also Threats to Validity")
    stat(sec, "test occurrences per held-out lemma, median", test_per_h.median(), code, task)
    stat(sec, "held-out lemmata with one test occurrence / |H|", (test_per_h == 1).mean(), code, task,
         "share")
    k = int(np.ceil(0.01 * len(test_per_h)))
    stat(sec, "test occurrences of the top 1% of held-out lemmata / |T|", test_per_h.iloc[:k].sum() / s["T"],
         code, task, "share", f"top {k} lemmata")
    core_share = gold[part == "core"].value_counts(normalize=True)
    reserve_share = gold[part == "reserve"].value_counts(normalize=True)
    stat(sec, "largest |reserve - core| label share", 100 * core_share.subtract(reserve_share, fill_value=0)
         .abs().max(), code, task, "pp")
    if task == "number":
        stat(sec, "smallest class count in T / min_per_class_number",
             min(s["T_by_class"].values()) / frozen_cfg["feasibility"]["min_per_class_number"], code, task, "ratio")
        stat(sec, "both classes in both training sets",
             all(len(s[f"train_{c}_by_class"]) == 2 for c in "AB"), code, task, "flag")

    # replay the held-out draw to name the lemma that crossed 20%; it must reproduce the frozen H
    seed = int.from_bytes(hashlib.sha256(f"{frozen_cfg['seed']}:{code}:{task}".encode()).digest()[:8], "big")
    candidates = np.array([l for l, n in count_of.items() if n >= 2], dtype=object)
    np.random.default_rng(seed).shuffle(candidates)
    target_T = round(split_cfg["test_fraction_target"] * N)
    floor_H = min(split_cfg["min_heldout_lemmas"], len(candidates))
    drawn, projected = [], 0
    for lem in candidates:
        if projected >= target_T and len(drawn) >= floor_H:
            break
        drawn.append(lem)
        projected += count_of[lem] // 2
    assert set(drawn) == held_out, f"{code}/{task}: the replayed draw differs from the frozen split"
    last = drawn[-1]
    stat(sec, "projected |T| / N before the last held-out lemma was added",
         (projected - count_of[last] // 2) / N, code, task, "share", f"last lemma added: {last}")
    stat(sec, "test occurrences of the last held-out lemma added / N", test_per_h.loc[last] / N, code, task,
         "share", last)

    sec = "Threats to Validity"
    lemmas_A = set(lemma[part.isin(["core", "seen"])])
    lemmas_B = set(lemma[part.isin(["core", "reserve"])])
    only_B = sorted(lemmas_B - lemmas_A)
    stat(sec, "condition-B lemma types absent from condition A / condition-B lemma types",
         len(only_B) / len(lemmas_B), code, task, "share")
    stat(sec, "single-occurrence lemmata among them", (per_lemma.loc[only_B] == 1).mean(), code, task, "share")
    if (code, task) == ("pl_pdb", "upos"):
        for block, keep in (("T", ["test"]), ("the seen block S", ["seen"]),
                            ("condition-A training", ["core", "seen"]),
                            ("condition-B training", ["core", "reserve"])):
            stat(sec, f"ADP share of {block}", (gold[part.isin(keep)] == "ADP").mean(), code, task, "share")
    if (code, task) == ("en_ewt", "upos"):
        part_test = lemma[(part == "test") & (gold == "PART")]
        stat(sec, "lemma types of the PART test occurrences", part_test.nunique(), code, task, "count",
             ", ".join(sorted(part_test.unique())))
        for cond, keep in (("A", ["core", "seen"]), ("B", ["core", "reserve"])):
            stat(sec, f"PART tokens in condition-{cond} training", ((gold == "PART") & part.isin(keep)).sum(),
                 code, task, "count")

# how gap and tests relate
for code, task in LT_CELLS:
    g, m = gap_tab.loc[(code, task)], mcn_tab.loc[(code, task)]
    b, c = m["b_A_correct_B_wrong"], m["c_A_wrong_B_correct"]
    n_test = main_res.loc[(code, task, "A"), "n_test"]
    sec = "Primary Result: Seen/Unseen Gap"
    stat(sec, "macro-F1 gap > accuracy gap", g["delta_macro_f1"] > g["delta_acc"], code, task, "flag")
    stat(sec, "width of the accuracy-gap interval", g["acc_ci_hi"] - g["acc_ci_lo"], code, task)
    stat(sec, "b / c", b / c, code, task, "ratio")
    stat(sec, "c / (b + c)", c / (b + c), code, task, "share")
    stat(sec, "(b + c) / |T|", (b + c) / n_test, code, task, "share")
    stat(sec, "accuracy gap = (b - c) / |T|", abs((b - c) / n_test - g["delta_acc"]) < 1e-12, code, task, "flag")
pl_upos = pcf[(pcf["language"] == "pl_pdb") & (pcf["task"] == "upos")].set_index("label")["delta_f1"]
stat("Primary Result: Seen/Unseen Gap", "SYM share of the macro-F1 gap", pl_upos["SYM"] / pl_upos.sum(),
     "pl_pdb", "upos", "share")

# task and control against control-label floor
for code, task in LT_CELLS:
    A, B = main_res.loc[(code, task, "A")], main_res.loc[(code, task, "B")]
    floor = A["control_majority_acc"]
    sec = "Selectivity"
    if task == "upos":
        stat(sec, "control accuracy A / control floor", A["control_acc"] / floor, code, task, "ratio")
        classes = pd.read_parquet(PRED_DIR / f"{code}_{task}_control.parquet",
                                  columns=["control_gold", "pred_A", "pred_B"])
        stat(sec, "UPOS tags realized among the control labels on T",
             len(set(classes["control_gold"]) | set(classes["pred_A"]) | set(classes["pred_B"])),
             code, task, "count", "footnote 9")
    else:
        stat(sec, "control accuracy A - control floor", A["control_acc"] - floor, code, task)
        stat(sec, "control macro-F1 A > its floor", A["control_f1"] > A["control_majority_f1"], code, task, "flag")
    stat(sec, "control accuracy B < control floor", B["control_acc"] < floor, code, task, "flag")
    stat(sec, "control macro-F1 B > its floor", B["control_f1"] > B["control_majority_f1"], code, task, "flag")
    stat(sec, "fall in control accuracy / fall in task accuracy, A to B",
         (A["control_acc"] - B["control_acc"]) / (A["accuracy"] - B["accuracy"]), code, task, "ratio")
    stat(sec, "accuracy selectivity B - A", B["selectivity_acc"] - A["selectivity_acc"], code, task)

# subword coefficient as odds ratios
for (code, task), g in coef.groupby(["language", "task"], sort=False):
    beta = g.set_index("term")["log_odds"]
    odds = np.exp(beta["z_n_subwords"])
    stat("Fragmentation", "error odds ratio per SD of n_subwords", odds, code, task, "ratio",
         f"1/OR = {1 / odds:.2f}" if odds < 1 else "")
    stat("Fragmentation", "char_length coefficient negative and larger than n_subwords in size",
         (beta["z_char_length"] < 0) and abs(beta["z_char_length"]) > abs(beta["z_n_subwords"]), code, task, "flag")

# where lemma-disjoint probes sit
for code in TB_CODES:
    A, B = main_res.loc[(code, "number", "A")], main_res.loc[(code, "number", "B")]
    floor = A["majority_test_acc"]
    minority = min(split_rep[f"{code}/number"]["T_by_class"].items(), key=lambda kv: kv[1])[0]
    sec = "Number Scores Near Ceiling"
    stat(sec, "(accuracy B - floor) / (1 - floor)", (B["accuracy"] - floor) / (1 - floor), code, "number", "share")
    stat(sec, "accuracy gap / (1 - floor)", (A["accuracy"] - B["accuracy"]) / (1 - floor), code, "number", "share")
    stat(sec, "F1 of the minority class in condition B",
         pcf[(pcf["language"] == code) & (pcf["task"] == "number") & (pcf["label"] == minority)]["f1_B"].iloc[0],
         code, "number", note=minority)

# control terms and per subword coefficient
upos_terms = {c: coef[(coef["language"] == c) & (coef["task"] == "upos")].set_index("term")["log_odds"]
              for c in TB_CODES}
sec = "Inconclusive Fragmentation Result"
for code in TB_CODES:
    stat(sec, "both continuous controls negative", (upos_terms[code][["z_char_length", "z_log_freq"]] < 0).all(),
         code, "upos", "flag")
stat(sec, "largest control magnitudes of the UPOS cells",
     all(abs(upos_terms["pl_pdb"][t]) >= max(abs(upos_terms[c][t]) for c in TB_CODES)
         for t in ("z_char_length", "z_log_freq")), "pl_pdb", "upos", "flag")
raw_sd = (pd.read_csv(RESULTS_DIR / "error_analysis_standardization.csv")
          .set_index(["language", "task", "predictor"])["raw_sd"])
all_coef = pd.concat([coef, pd.read_csv(RESULTS_DIR / "error_analysis_pooled.csv").assign(language="pooled")])
for (code, task), g in all_coef.groupby(["language", "task"], sort=False):
    beta = g.set_index("term")["log_odds"]["z_n_subwords"]
    stat(sec, "n_subwords log-odds per subword", beta / raw_sd[(code, task, "n_subwords")], code, task)

# Appendix D checks
sweep = pd.read_csv(RESULTS_DIR / "extension_layer_sweep.csv")
mlp = pd.read_csv(RESULTS_DIR / "extension_mlp_probe.csv")
sec = "Threats to Validity"
for code, task in LT_CELLS:
    f1_A, f1_B = main_res.loc[(code, task, "A"), "macro_f1"], main_res.loc[(code, task, "B"), "macro_f1"]
    for layer in (4, 11):
        r = sweep[(sweep["language"] == code) & (sweep["task"] == task) & (sweep["layer"] == layer)].iloc[0]
        stat(sec, f"layer {layer}: gap positive on both metrics", (r.delta_acc > 0) and (r.delta_f1 > 0),
             code, task, "flag")
        if task == "upos":
            stat(sec, f"layer {layer}: condition-B macro-F1 below layer 8", r.f1_B < f1_B, code, task, "flag")
        if (code, task) == ("pl_pdb", "upos"):   # the narrower Polish gap
            stat(sec, f"layer {layer}: macro-F1 gap narrower than at layer 8", r.delta_f1 < f1_A - f1_B,
                 code, task, "flag")
            stat(sec, f"layer {layer}: condition-A macro-F1 below layer 8", r.f1_A < f1_A, code, task, "flag")
for code, task in LT_CELLS:   # footnote 10: the MLP is not uniformly stronger
    f1_A, f1_B = main_res.loc[(code, task, "A"), "macro_f1"], main_res.loc[(code, task, "B"), "macro_f1"]
    r = mlp[(mlp["language"] == code) & (mlp["task"] == task)].iloc[0]
    stat(sec, "MLP macro-F1 below the linear probe in both conditions", (r.f1_A < f1_A) and (r.f1_B < f1_B),
         code, task, "flag", "footnote 10")

text_stat_tab = pd.DataFrame(text_stats)
text_stat_tab["section"] = pd.Categorical(text_stat_tab["section"], categories=SECTIONS, ordered=True)
text_stat_tab = text_stat_tab.sort_values("section", kind="stable").reset_index(drop=True)
text_stat_tab.to_csv(RESULTS_DIR / "text_statistics.csv", index=False)

SHORT = {"en_ewt": "en", "it_isdt": "it", "pl_pdb": "pl", "pooled": "pooled", "all": ""}


def where(r):
    return "/".join(x for x in (SHORT[r.language], "" if r.task == "all" else r.task) if x)


def shown(r, with_note=True):
    value = {"share": f"{100 * r.value:#.4g}".rstrip(".") + "%", "pp": f"{r.value:.2f} pp",
             "ratio": f"{r.value:.2f}", "count": f"{r.value:,.0f}",
             "GB": f"{r.value:.2f} GB"}.get(r.unit, f"{r.value:.4g}")
    note = f" ({r.note})" if with_note and r.note else ""
    return " ".join(x for x in (where(r), value) if x) + note


for section, sub in text_stat_tab.groupby("section", observed=True, sort=False):
    print(f"\n{section}")
    for statistic, rows in sub.groupby("statistic", sort=False):
        shared_note = rows["note"].iloc[0] if rows["note"].nunique() == 1 and len(rows) > 1 else ""
        head = f"  {statistic}" + (f" [{shared_note}]" if shared_note else "") + ": "
        if (rows["unit"] == "flag").all():   # stated for every cell, or for some (footnote 10)
            holds = [where(r) for r in rows.itertuples() if r.value]
            print(head + ("holds" if len(rows) == 1 and holds else "does not hold" if len(rows) == 1 else
                          f"holds in all {len(rows)}" if len(holds) == len(rows) else
                          f"holds in {len(holds)} of {len(rows)}" + (": " + ", ".join(holds) if holds else "")))
        else:
            print(head + ", ".join(shown(r, with_note=not shared_note) for r in rows.itertuples()))
print(f"\nwrote results/text_statistics.csv ({len(text_stat_tab)} rows, {text_stat_tab['statistic'].nunique()} statistics)")


Linguistic Preliminaries
  closed-class lemma types / lemma types: en/upos 1.156%, it/upos 1.445%, pl/upos 1.205%
  closed-class tokens / tokens: en/upos 38.32%, it/upos 34.57%, pl/upos 28.35%

Multilingual Encoders and XLM-R
  subwords of 'Kraków': 1 (▁Kraków)
  subwords of 'kraków': 2 (▁kra ków)

Tasks and Data
  three most frequent UPOS tags / tokens: en/upos 40.47% (NOUN, PUNCT, VERB), it/upos 47.65% (NOUN, PUNCT, DET), pl/upos 53.30% (NOUN, PUNCT, VERB)
  tokens of the least frequent UPOS tag: en/upos 329 (X), it/upos 26 (PART), pl/upos 21 (SYM)
  Sing / Number targets: en/number 76.32%, it/number 69.66%, pl/number 73.51%

Control Task and Selectivity
  lemma types with a control label: en/upos 16,526, en/number 6,036, it/upos 19,450, it/number 6,361, pl/upos 28,137, pl/number 11,061

Representation Extraction
  sentences whose rebuilt text differs from # text: en 1 (NO-BREAK SPACE in # text vs SPACE), it 0, pl 0
  targets whose span covers a differing character: en 0, it 0, pl 0